# TinyLlama Oracle LoRA V5 — NLP → SQL Audit Oracle

**Workflow optimise pour 90%+ de precision — v2.1 (correction SQL→FR)**

| Cellule | Description |
|---|---|
| 1 | Installation pip + redemarrage auto |
| 2 | Verification versions + GPU |
| 3 | Telechargement TinyLlama |
| 4 | Table ORACLE_AUDIT_TRAIL (500 lignes) |
| 5 | Dataset V4 : **7500** exemples FR<->SQL (7 blocs) — **CORRIGE** |
| 6 | Entrainement V5 : LoRA r=32, max_length=384, 5 epoques — **CORRIGE OOM** |
| 7 | Chargement V5 : inference + post-processing SQL semantique — template correct |
| 8 | Apercu table AUDIT + guide questions |
| 9 | **TEST FR→SQL** : scoring + execution reelle sur pandas |
| 10 | **TEST SQL→FR** : traduction resultats Oracle — **CORRIGE** (prompt matching) |
| 11 | Bilan global de performance |
| 12 | Guide deploiement production |

## Corrections V3 vs V2.0

| Probleme V2.0 | Solution V3 |
|---|---|
| Dataset charge : 4000/7500 (bloc7 SQL→FR tronque) | `raw.select(range(len(raw)))` force les 7500 |
| Prompt SQL→FR sans prefixe matching | Prefixe `"Traduis ce SQL Oracle en francais : "` exact |
| `generate()` appelle `post_process_sql` sur reponse FR | Deux fonctions separees : `generate_sql()` / `generate_fr()` |

> Executer cellule 1 seule puis attendre le redemarrage automatique.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — Installation des dependances                    ║
# ║  Executer SEULE puis attendre le redemarrage automatique     ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys

# Etape 1 : purger numpy corrompu
subprocess.run([sys.executable,"-m","pip","uninstall","-y","numpy"], capture_output=True)
subprocess.run([sys.executable,"-m","pip","cache","purge"], capture_output=True)

# Etape 2 : numpy compatible torch + tensorflow
subprocess.run([sys.executable,"-m","pip","install","-q","--no-cache-dir",
               "numpy>=2.0.0,<2.1.0"], check=True)

# Etape 3 : stack ML
subprocess.run([sys.executable,"-m","pip","install","-q","--no-cache-dir",
               "transformers>=4.40.0","peft>=0.10.0","accelerate>=0.28.0",
               "datasets>=2.19.0","torch>=2.2.0","huggingface_hub>=0.22.0",
               "pandas>=2.0.0","tabulate"], check=True)

print("\n✅ Installation terminee — redemarrage automatique...")
import os; os.kill(os.getpid(), 9)


In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — Verification des versions                       ║
# ╚══════════════════════════════════════════════════════════════╝
import sys, numpy as np, torch, transformers, peft, accelerate, datasets

print("="*55)
print("  ENVIRONNEMENT COLAB")
print("="*55)
print(f"  Python       : {sys.version.split()[0]}")
print(f"  NumPy        : {np.__version__}")
print(f"  PyTorch      : {torch.__version__}")
print(f"  Transformers : {transformers.__version__}")
print(f"  PEFT         : {peft.__version__}")
print(f"  Accelerate   : {accelerate.__version__}")
print(f"  Datasets     : {datasets.__version__}")
print("="*55)
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f"  GPU          : {dev.name}")
    print(f"  VRAM         : {dev.total_memory/1e9:.1f} GB")
    print("  ✅ GPU pret pour l'entrainement")
else:
    print("  ⚠️  Aucun GPU — activer dans Execution > Modifier le type d'execution")
print("="*55)


  ENVIRONNEMENT COLAB
  Python       : 3.12.12
  NumPy        : 2.0.2
  PyTorch      : 2.10.0+cu128
  Transformers : 5.0.0
  PEFT         : 0.18.1
  Accelerate   : 1.13.0
  Datasets     : 4.0.0
  GPU          : Tesla T4
  VRAM         : 15.6 GB
  ✅ GPU pret pour l'entrainement


In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — Telechargement TinyLlama                        ║
# ╚══════════════════════════════════════════════════════════════╝
from huggingface_hub import snapshot_download
import shutil, os

MODEL_DIR = "TinyLlama-1.1B-Chat-v1.0"

if not os.path.exists(MODEL_DIR):
    print("📥 Telechargement TinyLlama-1.1B-Chat-v1.0 (~2.2 GB)...")
    snapshot_download(repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                      local_dir=MODEL_DIR, local_dir_use_symlinks=False)
    print("✅ Modele telecharge.")
else:
    print(f"✅ Modele deja present : {MODEL_DIR}")

if not os.path.exists(MODEL_DIR + ".zip"):
    shutil.make_archive(MODEL_DIR, "zip", MODEL_DIR)
    print(f"📦 Archive : {MODEL_DIR}.zip")


📥 Telechargement TinyLlama-1.1B-Chat-v1.0 (~2.2 GB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

✅ Modele telecharge.
📦 Archive : TinyLlama-1.1B-Chat-v1.0.zip


In [3]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 4 — Table ORACLE_AUDIT_TRAIL (500 lignes)           ║
# ╚══════════════════════════════════════════════════════════════╝
import random, pandas as pd
from datetime import datetime, timedelta

random.seed(42)

OS_USERS   = ["oracle","sys","system","dbadmin","appuser","reports_user",
              "audit_user","dba_ops","backup_agent","monitor_svc"]
DB_USERS   = ["SYS","SYSTEM","SCOTT","HR","APP_USER","REPORT_USR",
              "AUDIT_USR","DBA_OPS","BACKUP_USR","MONITOR"]
SCHEMAS    = ["SYS","HR","SCOTT","APP_SCHEMA","AUDIT_SCHEMA"]
OBJECTS    = ["EMP","DEPT","SALGRADE","FACTURE","CLIENT","TRANSACTION",
              "AUDIT_LOG","V$SESSION","V$SQL","DBA_USERS","DBA_OBJECTS",
              "ALL_TABLES","PAYROLL","ORDERS","CUSTOMERS","PRODUCTS",
              "ACCOUNTS","JOURNAL","BUDGET","CONTRACTS"]
TERMINALS  = ["pts/0","pts/1","pts/2","pts/3","console","UNKNOWN","tty1"]
HOSTS      = ["srv-oracle-01","srv-oracle-02","app-server-01",
              "app-server-02","backup-srv","monitor-01","report-srv"]
PRIVILEGES = ["SELECT ANY TABLE","CREATE SESSION","DBA","SYSDBA",
              "ALTER SESSION","CREATE TABLE","INSERT ANY TABLE","UPDATE ANY TABLE"]
RETURNCODES = [0]*10 + [942,1017,1,955,1031,904]

SQL_TEXTS_SELECT = [
    "SELECT * FROM EMP WHERE DEPTNO=10",
    "SELECT COUNT(*) FROM ORDERS WHERE STATUS='PENDING'",
    "SELECT ENAME, SAL FROM EMP ORDER BY SAL DESC",
    "SELECT DISTINCT STATUS FROM V$SESSION",
    "SELECT * FROM DBA_OBJECTS WHERE OBJECT_TYPE='TABLE'",
]
SQL_TEXTS_OTHER = [
    "UPDATE FACTURE SET STATUT='PAYE' WHERE ID=2045",
    "DELETE FROM AUDIT_LOG WHERE TIMESTAMP < SYSDATE-90",
    "INSERT INTO ORDERS VALUES(:1,:2,:3,:4)",
    "CREATE TABLE TEMP_REPORT AS SELECT * FROM ORDERS WHERE ROWNUM<1000",
    "DROP TABLE TEMP_REPORT",
    "GRANT SELECT ON EMP TO REPORT_USR",
    "REVOKE DBA FROM SCOTT",
    "EXECUTE DBMS_STATS.GATHER_TABLE_STATS('HR','EMP')",
]
COMMENTS = [
    "Routine reporting query","Scheduled backup procedure",
    "Application login via connection pool","Automated monitoring script",
    "DBA maintenance task","End of day reconciliation",
    "","Emergency access - incident #4521","Performance audit",
    "Compliance check Q1-2026","","",
]

ACTION_POOL = (
    ["SELECT"] * 70 + ["LOGON"] * 10 + ["LOGOFF"] * 8 +
    ["INSERT"] * 3 + ["UPDATE"] * 3 + ["DELETE"] * 2 +
    ["EXECUTE"] * 2 + ["GRANT"] * 1 + ["REVOKE"] * 1
)

start = datetime(2026, 1, 1, 6, 0, 0)
rows = []
for i in range(1, 501):
    ts = start + timedelta(
        days=random.randint(0, 55),
        hours=random.randint(0, 17),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59)
    )
    action = random.choice(ACTION_POOL)
    rcode  = random.choice(RETURNCODES)
    is_select = action == "SELECT"
    is_conn   = action in ["LOGON","LOGOFF"]
    obj    = "" if is_conn else random.choice(OBJECTS)
    schema = "" if is_conn else random.choice(SCHEMAS)
    sql    = (random.choice(SQL_TEXTS_SELECT) if is_select
              else ("" if is_conn else random.choice(SQL_TEXTS_OTHER)))
    rows.append({
        "ID"             : i,
        "EVENT_TIMESTAMP": ts.strftime("%Y-%m-%d %H:%M:%S"),
        "OS_USERNAME"    : random.choice(OS_USERS),
        "DBUSERNAME"     : random.choice(DB_USERS),
        "USERHOST"       : random.choice(HOSTS),
        "TERMINAL"      : random.choice(TERMINALS),
        "SESSIONID"     : random.randint(100, 9999),
        "ACTION_NAME"   : action,
        "OBJECT_SCHEMA"  : schema,
        "OBJECT_NAME"    : obj,
        "SQL_TEXT"      : sql,
        "RETURNCODE"    : rcode,
        "PRIVILEGE_USED": random.choice(PRIVILEGES),
        "STATEMENT"     : i * 3 + random.randint(1,5),
        "COMMENT_TEXT"  : random.choice(COMMENTS),
    })

AUDIT_DF = pd.DataFrame(rows)
AUDIT_DF.to_csv("oracle_audit_trail.csv", index=False)

print("="*65)
print("  TABLE : ORACLE_AUDIT_TRAIL  (basee sur DBA_AUDIT_TRAIL)")
print("="*65)
print(f"  Lignes   : {len(AUDIT_DF)}")
print(f"  Colonnes : {len(AUDIT_DF.columns)}")
print(f"  Periode  : {AUDIT_DF.EVENT_TIMESTAMP.min()} -> {AUDIT_DF.EVENT_TIMESTAMP.max()}")
print()
print("  DISTRIBUTION DES ACTIONS :")
dist = AUDIT_DF.ACTION_NAME.value_counts()
total_rows = len(AUDIT_DF)
for action, count in dist.items():
    pct = count/total_rows*100
    bar = "█" * (count // 4)
    print(f"    {action:<20} {count:>4}  ({pct:>4.1f}%)  {bar}")
print()
print("  CODES RETOUR :")
rc = AUDIT_DF.RETURNCODE.value_counts().head(8)
for code, count in rc.items():
    label = "✅ Succes" if code == 0 else f"❌ ORA-{code:05d}"
    print(f"    {label:<25} {count:>4} occurrences")
print()
print("  APERCU (5 lignes) :")
print(AUDIT_DF[["ID","EVENT_TIMESTAMP","DBUSERNAME","ACTION_NAME",
                "OBJECT_NAME","RETURNCODE"]].head(5).to_string(index=False))
print()
print("✅ Table AUDIT — oracle_audit_trail.csv")


  TABLE : ORACLE_AUDIT_TRAIL  (basee sur DBA_AUDIT_TRAIL)
  Lignes   : 500
  Colonnes : 15
  Periode  : 2026-01-01 07:18:38 -> 2026-02-25 22:43:26

  DISTRIBUTION DES ACTIONS :
    SELECT                339  (67.8%)  ████████████████████████████████████████████████████████████████████████████████████
    LOGON                  47  ( 9.4%)  ███████████
    LOGOFF                 46  ( 9.2%)  ███████████
    UPDATE                 26  ( 5.2%)  ██████
    INSERT                 12  ( 2.4%)  ███
    DELETE                 11  ( 2.2%)  ██
    GRANT                   8  ( 1.6%)  ██
    EXECUTE                 6  ( 1.2%)  █
    REVOKE                  5  ( 1.0%)  █

  CODES RETOUR :
    ✅ Succes                   309 occurrences
    ❌ ORA-00904                 37 occurrences
    ❌ ORA-01017                 32 occurrences
    ❌ ORA-00955                 31 occurrences
    ❌ ORA-00001                 31 occurrences
    ❌ ORA-00942                 30 occurrences
    ❌ ORA-01031                 3

In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 5 — Dataset V4 : 7500 exemples                      ║
# ║                                                               ║
# ║  NOUVEAUTES V4 :                                              ║
# ║  - Colonnes réelles : DBUSERNAME, EVENT_TIMESTAMP,            ║
# ║    USERHOST, ID, OBJECT_NAME, OBJECT_SCHEMA                   ║
# ║  - Filtre temporel : TRUNC(EVENT_TIMESTAMP)                   ║
# ║  - Nouvelles ACTION_NAME : ALTER TABLE/USER, DROP USER/TABLE  ║
# ║    CREATE USER/TABLE, GRANT, REVOKE, TRUNCATE, RENAME,        ║
# ║    COMMENT, AUDIT, NOAUDIT, EXECUTE, INDEX, LOCK TABLE        ║
# ║  - Conserve toutes les actions V3 : LOGON/LOGOFF/SELECT/...   ║
# ╚══════════════════════════════════════════════════════════════╝
import pandas as pd, random
random.seed(42)

DB_USERS = ["SYSTEM","VROMUALD","SYS","HR","APP_USER","REPORT_USR",
            "AUDIT_USR","DBA_OPS","BACKUP_USR","MONITOR","SCOTT","ETL_USER"]
OBJECTS  = ["EMPLOYEES","VROMUALD","CLIENTS","TRANSACTIONS","ORDERS",
            "ACCOUNTS","PAYROLL","JOURNAL","CONTRACTS","BUDGET",
            "PRODUCTS","SUPPLIERS","INVOICES","USERS","PAYMENTS",
            "AUDIT_LOG","SESSIONS","ALERTS","DBA_USERS","V$SESSION"]

# Table réelle utilisée dans tous les exemples
TABLE = "ORACLE_AUDIT_TRAIL"

# ══════════════════════════════════════════════════════════════
# BLOC 1 — Utilisateurs existants -> DBA_USERS — 400 ex
# ══════════════════════════════════════════════════════════════
bloc1 = []
b1_base = [
    ("Combien d'utilisateurs existent dans la base Oracle ?",
     "SELECT COUNT(*) AS NB_UTILISATEURS FROM DBA_USERS;"),
    ("Donne-moi le nombre d'utilisateurs crees dans Oracle.",
     "SELECT COUNT(*) AS NB_UTILISATEURS FROM DBA_USERS;"),
    ("Quel est le nombre total de comptes Oracle ?",
     "SELECT COUNT(*) AS NB_COMPTES FROM DBA_USERS;"),
    ("Combien de comptes utilisateurs Oracle existent ?",
     "SELECT COUNT(*) AS NB_UTILISATEURS FROM DBA_USERS;"),
    ("Liste tous les utilisateurs Oracle existants.",
     "SELECT USERNAME, ACCOUNT_STATUS, CREATED FROM DBA_USERS ORDER BY USERNAME;"),
    ("Quels utilisateurs Oracle sont actifs ?",
     "SELECT USERNAME, ACCOUNT_STATUS FROM DBA_USERS WHERE ACCOUNT_STATUS='OPEN' ORDER BY USERNAME;"),
    ("Affiche tous les comptes Oracle avec leur statut.",
     "SELECT USERNAME, ACCOUNT_STATUS, CREATED, LOCK_DATE FROM DBA_USERS ORDER BY USERNAME;"),
    ("Combien d'utilisateurs Oracle ont un compte actif ?",
     "SELECT COUNT(*) AS NB_ACTIFS FROM DBA_USERS WHERE ACCOUNT_STATUS='OPEN';"),
    ("Liste les utilisateurs Oracle verrouilles.",
     "SELECT USERNAME, LOCK_DATE FROM DBA_USERS WHERE ACCOUNT_STATUS='LOCKED' ORDER BY LOCK_DATE;"),
    ("Qui sont les utilisateurs Oracle de la base ?",
     "SELECT USERNAME, ACCOUNT_STATUS, CREATED FROM DBA_USERS ORDER BY USERNAME;"),
    ("Nombre total d'utilisateurs Oracle en base.",
     "SELECT COUNT(*) AS NB_UTILISATEURS FROM DBA_USERS;"),
    ("Combien d'actions ont ete enregistrees dans l'audit Oracle ?",
     "SELECT COUNT(*) AS NB_ACTIONS FROM ORACLE_AUDIT_TRAIL;"),
    ("Nombre de lignes dans la table d'audit Oracle.",
     "SELECT COUNT(*) AS NB_LIGNES FROM ORACLE_AUDIT_TRAIL;"),
]
for q, sql in b1_base:
    bloc1.append({"instruction": q, "output": sql})
    for pfx in ["Oracle DBA : ","Question DBA : ","Admin Oracle : ","Requete admin : "]:
        bloc1.append({"instruction": pfx + q, "output": sql})
while len(bloc1) < 400:
    ex = random.choice(bloc1[:65])
    pfx = random.choice(["Audit : ","DBA Oracle : ","Analyse : ","Bilan : "])
    bloc1.append({"instruction": pfx + ex["instruction"].lower().capitalize(), "output": ex["output"]})
bloc1 = bloc1[:400]

# ══════════════════════════════════════════════════════════════
# BLOC 2 — Utilisateur le plus / moins actif — 400 ex
# Colonnes réelles : DBUSERNAME
# ══════════════════════════════════════════════════════════════
bloc2 = []
b2_base = [
    ("Qui est l'utilisateur le plus actif dans la base Oracle ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Quel utilisateur a fait le plus d'operations dans l'audit Oracle ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Affiche le top 5 des utilisateurs par nombre d'actions Oracle.",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 5 ROWS ONLY;"),
    ("Classe les utilisateurs Oracle par nombre total d'actions.",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DBUSERNAME ORDER BY NB DESC;"),
    ("Qui touche le plus a la base Oracle ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Qui fait le moins de choses dans la base Oracle ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DBUSERNAME ORDER BY NB ASC FETCH FIRST 1 ROWS ONLY;"),
    ("Quel utilisateur est le moins actif dans la base ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DBUSERNAME ORDER BY NB ASC FETCH FIRST 1 ROWS ONLY;"),
    ("Classe les utilisateurs Oracle du moins actif au plus actif.",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DBUSERNAME ORDER BY NB ASC;"),
    ("Top 5 des utilisateurs les moins actifs.",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DBUSERNAME ORDER BY NB ASC FETCH FIRST 5 ROWS ONLY;"),
    ("Donne-moi le classement des utilisateurs par activite.",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DBUSERNAME ORDER BY NB DESC;"),
]
for q, sql in b2_base:
    bloc2.append({"instruction": q, "output": sql})
    for pfx in ["Oracle audit : ","Question : ","Admin : ","Analyse securite : "]:
        bloc2.append({"instruction": pfx + q, "output": sql})
while len(bloc2) < 400:
    ex = random.choice(bloc2[:60])
    pfx = random.choice(["Audit Oracle : ","Bilan : ","Top utilisateurs : ","Activite : "])
    bloc2.append({"instruction": pfx + ex["instruction"].lower().capitalize(), "output": ex["output"]})
bloc2 = bloc2[:400]

# ══════════════════════════════════════════════════════════════
# BLOC 3 — Derniere personne / tracabilite — 400 ex
# Colonnes réelles : DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP, OBJECT_NAME
# ══════════════════════════════════════════════════════════════
bloc3 = []
b3_base = [
    ("Qui a touche la table EMPLOYEES en dernier ?",
     "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='EMPLOYEES' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Quelle est la derniere personne a avoir modifie VROMUALD ?",
     "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='VROMUALD' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Qui a fait la derniere action sur CLIENTS ?",
     "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='CLIENTS' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Dernier acces a la table TRANSACTIONS ?",
     "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='TRANSACTIONS' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Qui a touche ORDERS en dernier dans l'audit ?",
     "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='ORDERS' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Derniere action enregistree sur PAYROLL ?",
     "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='PAYROLL' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Qui est le dernier a avoir interagi avec ACCOUNTS ?",
     "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='ACCOUNTS' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Quel utilisateur a acces en dernier a CONTRACTS ?",
     "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='CONTRACTS' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 1 ROWS ONLY;"),
]
for q, sql in b3_base:
    bloc3.append({"instruction": q, "output": sql})
    for pfx in ["Tracabilite : ","Audit securite : ","Oracle : ","Question : "]:
        bloc3.append({"instruction": pfx + q, "output": sql})
while len(bloc3) < 400:
    obj = random.choice(["EMPLOYEES","VROMUALD","CLIENTS","TRANSACTIONS","ORDERS",
                         "ACCOUNTS","PAYROLL","JOURNAL","CONTRACTS","BUDGET",
                         "PRODUCTS","INVOICES","USERS","PAYMENTS","SUPPLIERS"])
    q = random.choice([
        f"Derniere personne a avoir touche {obj} ?",
        f"Qui a touche {obj} en dernier dans l'audit ?",
        f"Quel utilisateur a acces en dernier a {obj} ?",
        f"Derniere action sur la table {obj} ?",
        f"Qui est le dernier a avoir interagi avec {obj} ?",
    ])
    sql = f"SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='{obj}' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 1 ROWS ONLY;"
    bloc3.append({"instruction": random.choice(["Audit : ","Tracabilite : ","Oracle : "]) + q, "output": sql})
bloc3 = bloc3[:400]

# ══════════════════════════════════════════════════════════════
# BLOC 4 — Dernieres N actions — 300 ex
# Colonnes réelles : ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME
# ══════════════════════════════════════════════════════════════
bloc4 = []
b4_base = [
    ("Donne-moi les 5 dernieres actions enregistrees dans la base.",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 5 ROWS ONLY;"),
    ("Affiche les 10 dernieres entrees d'audit Oracle.",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 10 ROWS ONLY;"),
    ("Quelles sont les 3 dernieres operations dans l'audit ?",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 3 ROWS ONLY;"),
    ("Derniere action enregistree dans l'audit Oracle.",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Qu'est-ce qui s'est passe en dernier dans la base ?",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 10 ROWS ONLY;"),
    ("Montre-moi les 20 derniers evenements d'audit Oracle.",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 20 ROWS ONLY;"),
]
for q, sql in b4_base:
    bloc4.append({"instruction": q, "output": sql})
    for pfx in ["Oracle : ","Audit : ","Rapport : ","DBA : "]:
        bloc4.append({"instruction": pfx + q, "output": sql})
while len(bloc4) < 300:
    n = random.choice([1, 3, 5, 10, 20, 50])
    q = random.choice([
        f"Affiche les {n} dernieres actions dans l'audit Oracle.",
        f"Les {n} derniers evenements d'audit Oracle.",
        f"Donne les {n} dernieres entrees de la table d'audit.",
        f"Montre les {n} dernieres operations enregistrees.",
    ])
    sql = f"SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST {n} ROWS ONLY;"
    bloc4.append({"instruction": q, "output": sql})
bloc4 = bloc4[:300]

# ══════════════════════════════════════════════════════════════
# BLOC 5 — COUNT DISTINCT + stats — 300 ex
# ══════════════════════════════════════════════════════════════
bloc5 = []
b5_base = [
    ("Combien de tables distinctes ont ete accedees dans l'audit Oracle ?",
     "SELECT COUNT(DISTINCT OBJECT_NAME) AS NB_TABLES FROM ORACLE_AUDIT_TRAIL;"),
    ("Quel est le nombre d'utilisateurs distincts dans la trace d'audit ?",
     "SELECT COUNT(DISTINCT DBUSERNAME) AS NB_USERS FROM ORACLE_AUDIT_TRAIL;"),
    ("Quel utilisateur a acces au plus grand nombre de tables differentes ?",
     "SELECT DBUSERNAME, COUNT(DISTINCT OBJECT_NAME) AS NB_TABLES FROM ORACLE_AUDIT_TRAIL GROUP BY DBUSERNAME ORDER BY NB_TABLES DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Affiche la distribution de toutes les actions Oracle dans l'audit.",
     "SELECT ACTION_NAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY ACTION_NAME ORDER BY NB DESC;"),
    ("Resume global : total evenements dans l'audit Oracle.",
     "SELECT COUNT(*) AS TOTAL FROM ORACLE_AUDIT_TRAIL;"),
    ("Quels utilisateurs ont acces aux objets systeme sensibles ?",
     "SELECT DISTINCT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME IN ('DBA_USERS','V$SESSION') GROUP BY DBUSERNAME ORDER BY NB DESC;"),
]
for q, sql in b5_base:
    bloc5.append({"instruction": q, "output": sql})
    for pfx in ["Stats Oracle : ","Rapport : ","Analyse : ","Bilan securite : ","COUNT DISTINCT : "]:
        bloc5.append({"instruction": pfx + q, "output": sql})
while len(bloc5) < 300:
    ex = random.choice(bloc5[:36])
    pfx = random.choice(["Stats Oracle : ","Rapport : ","Bilan securite : "])
    bloc5.append({"instruction": pfx + ex["instruction"].lower().capitalize(), "output": ex["output"]})
bloc5 = bloc5[:300]

# ══════════════════════════════════════════════════════════════
# BLOC 6 — SELECT simples + securite — 400 ex
# ══════════════════════════════════════════════════════════════
bloc6 = []
b6_base = [
    ("Quels utilisateurs Oracle ont effectue des SELECT dans l'audit ?",
     "SELECT DISTINCT DBUSERNAME FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' ORDER BY DBUSERNAME;"),
    ("Combien de SELECT ont ete enregistres dans l'audit Oracle ?",
     "SELECT COUNT(*) AS NB_SELECT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT';"),
    ("Quel utilisateur Oracle a fait le plus de SELECT ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Y a-t-il eu des connexions nocturnes apres 22h ?",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, USERHOST FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' AND TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= 22 ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Detecte les tentatives de connexion echouees.",
     "SELECT DBUSERNAME, USERHOST, COUNT(*) AS NB_ECHECS FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' AND ACTION_NAME='LOGON' GROUP BY DBUSERNAME, USERHOST ORDER BY NB_ECHECS DESC;"),
    ("Supprime tous les enregistrements d'audit.",
     "-- REQUETE REFUSEE : les operations DELETE ne sont pas autorisees. Ce systeme est en lecture seule."),
    ("Tronque la table d'audit Oracle.",
     "-- REQUETE REFUSEE : TRUNCATE non autorise. La table d'audit est immuable."),
    ("Combien de connexions LOGON ont ete enregistrees ?",
     "SELECT COUNT(*) AS NB_CONNEXIONS FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON';"),
]
for q, sql in b6_base:
    bloc6.append({"instruction": q, "output": sql})
    for pfx in ["Oracle audit : ","Requete : ","Analyse : "]:
        bloc6.append({"instruction": pfx + q, "output": sql})
while len(bloc6) < 400:
    ex = random.choice(bloc6[:32])
    pfx = random.choice(["Question SQL : ","Audit Oracle : ","Rapport : "])
    bloc6.append({"instruction": pfx + ex["instruction"].lower().capitalize(), "output": ex["output"]})
bloc6 = bloc6[:400]

# ══════════════════════════════════════════════════════════════
# BLOC 7 — SQL -> FR — 400 ex
# ══════════════════════════════════════════════════════════════
bloc7 = []
b7_base = [
    ("SELECT COUNT(*) AS NB_UTILISATEURS FROM DBA_USERS;",
     "La base Oracle contient un certain nombre d'utilisateurs stockes dans la vue systeme DBA_USERS."),
    ("SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='EMPLOYEES' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 1 ROWS ONLY;",
     "La derniere personne a avoir touche la table EMPLOYEES est indiquee avec le type d'action et l'horodatage."),
    ("SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 5 ROWS ONLY;",
     "Les 5 dernieres actions enregistrees dans l'audit Oracle sont affichees de la plus recente a la plus ancienne."),
    ("SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DBUSERNAME ORDER BY NB DESC;",
     "Le volume total d'activite par utilisateur Oracle est affiche, du plus actif au moins actif."),
    ("SELECT ACTION_NAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY ACTION_NAME ORDER BY NB DESC;",
     "La distribution de toutes les actions Oracle enregistrees dans l'audit est affichee."),
    ("SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;",
     "L'utilisateur Oracle ayant effectue le plus de connexions est identifie."),
    ("SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;",
     "L'utilisateur Oracle ayant effectue le plus de lectures est identifie."),
    ("SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-1) ORDER BY EVENT_TIMESTAMP DESC;",
     "L'activite enregistree hier dans l'audit Oracle est affichee, de la plus recente a la plus ancienne."),
    ("SELECT DBUSERNAME, ACTION_NAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-7) GROUP BY DBUSERNAME, ACTION_NAME ORDER BY NB DESC;",
     "Le detail des actions par utilisateur sur les 7 derniers jours est affiche."),
]
prefixes_sqlfr = [
    "Traduis ce SQL Oracle en francais : ",
    "Explique cette requete Oracle : ",
    "Que retourne cette requete sur ORACLE_AUDIT_TRAIL ? ",
    "Decris en francais ce que fait : ",
    "Explique pour le responsable securite : ",
]
for sql, expl in b7_base:
    for pfx in prefixes_sqlfr:
        bloc7.append({"instruction": pfx + sql, "output": expl})
while len(bloc7) < 400:
    ex = random.choice(bloc7[:45])
    pfx = random.choice(prefixes_sqlfr)
    bloc7.append({"instruction": pfx + ex["instruction"].split(" : ",1)[-1] if " : " in ex["instruction"] else pfx + ex["instruction"],
                  "output": ex["output"]})
bloc7 = bloc7[:400]

# ══════════════════════════════════════════════════════════════
# BLOC 8 — ACTION_NAME classiques : LOGON/SELECT/INSERT/UPDATE/DELETE/LOGOFF — 500 ex
# Colonnes réelles : DBUSERNAME, EVENT_TIMESTAMP
# ══════════════════════════════════════════════════════════════
bloc8 = []
b8_base = [
    # LOGON
    ("Qui est l'utilisateur qui a effectue le plus de connexions ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB_CONNEXIONS FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' GROUP BY DBUSERNAME ORDER BY NB_CONNEXIONS DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Combien de connexions ont ete enregistrees dans l'audit Oracle ?",
     "SELECT COUNT(*) AS NB_CONNEXIONS FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON';"),
    ("Affiche tous les utilisateurs qui se sont connectes.",
     "SELECT DISTINCT DBUSERNAME FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' ORDER BY DBUSERNAME;"),
    ("Liste les 10 dernieres connexions Oracle.",
     "SELECT DBUSERNAME, EVENT_TIMESTAMP, USERHOST FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 10 ROWS ONLY;"),
    ("Qui s'est connecte a la base ces derniers jours ?",
     "SELECT DISTINCT DBUSERNAME, MAX(EVENT_TIMESTAMP) AS DERNIERE_CONNEXION FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' GROUP BY DBUSERNAME ORDER BY DERNIERE_CONNEXION DESC;"),
    # SELECT
    ("Quel utilisateur a effectue le plus de lectures dans la base ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB_SELECT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DBUSERNAME ORDER BY NB_SELECT DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Combien de lectures ont ete faites dans la base ?",
     "SELECT COUNT(*) AS NB_SELECT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT';"),
    ("Qui consulte le plus la base de donnees ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    # INSERT
    ("Qui a fait le plus d'ajouts dans la base Oracle ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='INSERT' GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Combien d'insertions ont ete faites dans l'audit Oracle ?",
     "SELECT COUNT(*) AS NB_INSERT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='INSERT';"),
    # UPDATE
    ("Combien de modifications ont ete enregistrees dans l'audit ?",
     "SELECT COUNT(*) AS NB_UPDATE FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='UPDATE';"),
    ("Qui a effectue le plus de modifications dans la base Oracle ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='UPDATE' GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    # DELETE
    ("Qui a supprime le plus de donnees dans la base Oracle ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='DELETE' GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Combien de suppressions ont ete faites dans l'audit Oracle ?",
     "SELECT COUNT(*) AS NB_DELETE FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='DELETE';"),
    # LOGOFF
    ("Combien de deconnexions ont ete enregistrees dans l'audit ?",
     "SELECT COUNT(*) AS NB_LOGOFF FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGOFF';"),
    ("Affiche les dernieres deconnexions Oracle.",
     "SELECT DBUSERNAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGOFF' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 10 ROWS ONLY;"),
]
for q, sql in b8_base:
    bloc8.append({"instruction": q, "output": sql})
    for pfx in ["Oracle audit : ","Securite : ","Analyse : ","DBA : ","Question : "]:
        bloc8.append({"instruction": pfx + q, "output": sql})
while len(bloc8) < 500:
    ex = random.choice(bloc8[:96])
    pfx = random.choice(["Audit Oracle : ","Question SQL : ","Rapport : ","Stats : "])
    bloc8.append({"instruction": pfx + ex["instruction"].lower().capitalize(), "output": ex["output"]})
bloc8 = bloc8[:500]

# ══════════════════════════════════════════════════════════════
# BLOC 9 — NOUVELLES ACTION_NAME DDL/DCL — 600 ex
# ALTER TABLE/USER, DROP USER/TABLE, CREATE USER/TABLE, GRANT, REVOKE,
# TRUNCATE, RENAME, COMMENT, AUDIT, NOAUDIT, EXECUTE, INDEX, LOCK TABLE
# ══════════════════════════════════════════════════════════════
bloc9 = []
b9_base = [
    # ALTER TABLE
    ("Qui a modifie la structure d'une table dans la base ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='ALTER TABLE' ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Combien d'operations ALTER TABLE ont ete enregistrees ?",
     "SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='ALTER TABLE';"),
    ("Qui a fait le plus d'alterations de tables ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='ALTER TABLE' GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Quelles tables ont ete modifiees dans la structure ?",
     "SELECT DISTINCT OBJECT_NAME, DBUSERNAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='ALTER TABLE' ORDER BY EVENT_TIMESTAMP DESC;"),
    # ALTER USER
    ("Qui a modifie des comptes utilisateurs dans la base ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='ALTER USER' ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Combien de modifications de comptes ont ete faites ?",
     "SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='ALTER USER';"),
    ("Quels utilisateurs ont ete modifies recemment ?",
     "SELECT OBJECT_NAME, DBUSERNAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='ALTER USER' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 10 ROWS ONLY;"),
    # DROP USER
    ("Y a-t-il eu des suppressions de comptes utilisateurs ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='DROP USER' ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Combien de comptes ont ete supprimes dans la base ?",
     "SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='DROP USER';"),
    ("Qui a supprime des comptes utilisateurs Oracle ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='DROP USER' GROUP BY DBUSERNAME ORDER BY NB DESC;"),
    # DROP TABLE
    ("Y a-t-il eu des suppressions de tables dans la base ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='DROP TABLE' ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Combien de tables ont ete supprimees ?",
     "SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='DROP TABLE';"),
    ("Qui a supprime des tables dans la base Oracle ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='DROP TABLE' GROUP BY DBUSERNAME ORDER BY NB DESC;"),
    # CREATE USER
    ("Qui a cree des comptes utilisateurs dans la base ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='CREATE USER' ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Combien de comptes ont ete crees dans la base Oracle ?",
     "SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='CREATE USER';"),
    ("Quels nouveaux comptes ont ete crees recemment ?",
     "SELECT OBJECT_NAME, DBUSERNAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='CREATE USER' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 10 ROWS ONLY;"),
    # CREATE TABLE
    ("Qui a cree des tables dans la base Oracle ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='CREATE TABLE' ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Combien de tables ont ete creees dans la base ?",
     "SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='CREATE TABLE';"),
    ("Quelles nouvelles tables ont ete creees ?",
     "SELECT OBJECT_NAME, DBUSERNAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='CREATE TABLE' ORDER BY EVENT_TIMESTAMP DESC;"),
    # GRANT
    ("Qui a accorde des privileges dans la base Oracle ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='GRANT' ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Combien de GRANT ont ete enregistres dans l'audit ?",
     "SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='GRANT';"),
    ("Qui a le plus accorde de privileges ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='GRANT' GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    # REVOKE
    ("Qui a retire des privileges dans la base Oracle ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='REVOKE' ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Combien de REVOKE ont ete enregistres ?",
     "SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='REVOKE';"),
    # TRUNCATE
    ("Y a-t-il eu des TRUNCATE dans la base Oracle ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='TRUNCATE TABLE' ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a vide des tables dans la base Oracle ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='TRUNCATE TABLE' ORDER BY EVENT_TIMESTAMP DESC;"),
    # EXECUTE
    ("Qui a execute des procedures stockees dans la base ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='EXECUTE' ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Combien d'executions de procedures ont ete enregistrees ?",
     "SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='EXECUTE';"),
    # LOCK TABLE
    ("Y a-t-il eu des verrous de tables dans la base ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOCK TABLE' ORDER BY EVENT_TIMESTAMP DESC;"),
    # Combinaisons DDL
    ("Affiche toutes les operations de creation et suppression dans la base.",
     "SELECT DBUSERNAME, ACTION_NAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME IN ('CREATE TABLE','DROP TABLE','CREATE USER','DROP USER') ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a fait des modifications structurelles dans la base ?",
     "SELECT DBUSERNAME, ACTION_NAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME IN ('ALTER TABLE','ALTER USER','DROP TABLE','DROP USER','CREATE TABLE','CREATE USER') ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Affiche toutes les operations d'administration des droits.",
     "SELECT DBUSERNAME, ACTION_NAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME IN ('GRANT','REVOKE') ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a fait des changements dangereux dans la base ?",
     "SELECT DBUSERNAME, ACTION_NAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME IN ('DROP TABLE','DROP USER','TRUNCATE TABLE','REVOKE') ORDER BY EVENT_TIMESTAMP DESC;"),
]
for q, sql in b9_base:
    bloc9.append({"instruction": q, "output": sql})
    for pfx in ["Oracle audit : ","Securite : ","Analyse : ","DBA : ","Question : "]:
        bloc9.append({"instruction": pfx + q, "output": sql})
while len(bloc9) < 600:
    ex = random.choice(bloc9[:198])
    pfx = random.choice(["DDL : ","DCL : ","Admin Oracle : ","Audit : ","Rapport : "])
    bloc9.append({"instruction": pfx + ex["instruction"].lower().capitalize(), "output": ex["output"]})
bloc9 = bloc9[:600]

# ══════════════════════════════════════════════════════════════
# BLOC 10 — OBJ_NAME textuel — 400 ex
# Colonnes réelles : OBJECT_NAME, DBUSERNAME, EVENT_TIMESTAMP
# ══════════════════════════════════════════════════════════════
bloc10 = []
obj_templates = [
    ("Qui a accede a la table {obj} dans l'audit Oracle ?",
     "SELECT DISTINCT DBUSERNAME FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='{obj}' ORDER BY DBUSERNAME;"),
    ("Combien d'actions ont ete effectuees sur {obj} ?",
     "SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='{obj}';"),
    ("Quelles sont les dernieres actions sur la table {obj} ?",
     "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='{obj}' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 10 ROWS ONLY;"),
    ("Quel utilisateur a effectue le plus d'actions sur {obj} ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='{obj}' GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Affiche toutes les lectures sur la table {obj}.",
     "SELECT DBUSERNAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='{obj}' AND ACTION_NAME='SELECT' ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui touche a {obj} dans la base ?",
     "SELECT DISTINCT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='{obj}' GROUP BY DBUSERNAME ORDER BY NB DESC;"),
    ("Quelles sont les deux personnes qui touchent le plus a {obj} ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='{obj}' GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 2 ROWS ONLY;"),
]
obj_names = ["EMPLOYEES","VROMUALD","CLIENTS","TRANSACTIONS","ORDERS",
             "ACCOUNTS","PAYROLL","JOURNAL","CONTRACTS","BUDGET",
             "PRODUCTS","SUPPLIERS","INVOICES","USERS","PAYMENTS",
             "AUDIT_LOG","SESSIONS","ALERTS","DBA_USERS","V$SESSION"]
for obj in obj_names:
    for tmpl_q, tmpl_sql in obj_templates:
        q   = tmpl_q.format(obj=obj)
        sql = tmpl_sql.format(obj=obj)
        bloc10.append({"instruction": q, "output": sql})
while len(bloc10) < 400:
    obj = random.choice(obj_names)
    tmpl_q, tmpl_sql = random.choice(obj_templates)
    q   = tmpl_q.format(obj=obj)
    sql = tmpl_sql.format(obj=obj)
    pfx = random.choice(["Securite : ","Tracabilite : ","DBA : ","Rapport : "])
    bloc10.append({"instruction": pfx + q.lower().capitalize(), "output": sql})
bloc10 = bloc10[:400]

# ══════════════════════════════════════════════════════════════
# BLOC 11 — Notions de temps — 600 ex
# Filtre réel : TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-N)
# ══════════════════════════════════════════════════════════════
bloc11 = []
b11_base = [
    # AUJOURD'HUI
    ("Qu'est-ce qui s'est passe aujourd'hui dans la base Oracle ?",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE) ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a touche la base aujourd'hui ?",
     "SELECT DISTINCT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE) GROUP BY DBUSERNAME ORDER BY NB DESC;"),
    ("Combien d'actions ont ete enregistrees aujourd'hui ?",
     "SELECT COUNT(*) AS NB_AUJOURD_HUI FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE);"),
    ("Quelles actions ont ete faites aujourd'hui ?",
     "SELECT ACTION_NAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE) GROUP BY ACTION_NAME ORDER BY NB DESC;"),
    # HIER
    ("Qu'est-ce qui s'est passe dans la base hier ?",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-1) ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a acces a la base hier ?",
     "SELECT DISTINCT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-1) GROUP BY DBUSERNAME ORDER BY NB DESC;"),
    ("Combien d'actions ont ete faites hier dans la base Oracle ?",
     "SELECT COUNT(*) AS NB_HIER FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-1);"),
    ("Qui a touche la base hier ?",
     "SELECT DISTINCT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-1) GROUP BY DBUSERNAME ORDER BY NB DESC;"),
    ("Montre-moi ce qui s'est passe hier dans la base.",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-1) ORDER BY EVENT_TIMESTAMP DESC;"),
    # AVANT-HIER
    ("Qu'est-ce qui s'est passe avant-hier dans la base Oracle ?",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-2) ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a acces a la base avant-hier ?",
     "SELECT DISTINCT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-2) GROUP BY DBUSERNAME ORDER BY NB DESC;"),
    # CETTE SEMAINE
    ("Qu'est-ce qui s'est passe cette semaine dans la base Oracle ?",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-7) ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a touche la base cette semaine ?",
     "SELECT DISTINCT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-7) GROUP BY DBUSERNAME ORDER BY NB DESC;"),
    ("Combien d'actions ont ete enregistrees cette semaine ?",
     "SELECT COUNT(*) AS NB_SEMAINE FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-7);"),
    ("Qui a fait quoi cette semaine dans la base ?",
     "SELECT DBUSERNAME, ACTION_NAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-7) GROUP BY DBUSERNAME, ACTION_NAME ORDER BY DBUSERNAME, NB DESC;"),
    # CE MOIS
    ("Qu'est-ce qui s'est passe ce mois-ci dans la base Oracle ?",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP,'MM') = TRUNC(SYSDATE,'MM') ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a acces a la base ce mois-ci ?",
     "SELECT DISTINCT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP,'MM') = TRUNC(SYSDATE,'MM') GROUP BY DBUSERNAME ORDER BY NB DESC;"),
    ("Combien d'actions ont ete faites ce mois-ci ?",
     "SELECT COUNT(*) AS NB_MOIS FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP,'MM') = TRUNC(SYSDATE,'MM');"),
    # LE MOIS PASSE
    ("Qu'est-ce qui s'est passe le mois dernier dans la base Oracle ?",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP,'MM') = TRUNC(ADD_MONTHS(SYSDATE,-1),'MM') ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a touche la base le mois passe ?",
     "SELECT DISTINCT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP,'MM') = TRUNC(ADD_MONTHS(SYSDATE,-1),'MM') GROUP BY DBUSERNAME ORDER BY NB DESC;"),
    # LA NUIT
    ("Y a-t-il eu des activites la nuit dans la base Oracle ?",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL WHERE TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= 22 OR TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) <= 5 ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a touche la base la nuit ?",
     "SELECT DISTINCT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= 22 OR TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) <= 5 GROUP BY DBUSERNAME ORDER BY NB DESC;"),
    ("Activite nocturne apres 22h dans l'audit Oracle.",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL WHERE TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= 22 ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Quelles connexions ont eu lieu la nuit ?",
     "SELECT DBUSERNAME, EVENT_TIMESTAMP, USERHOST FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' AND (TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= 22 OR TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) <= 5) ORDER BY EVENT_TIMESTAMP DESC;"),
    # LE MATIN
    ("Qui s'est connecte ce matin dans la base Oracle ?",
     "SELECT DBUSERNAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' AND TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE) AND TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= 6 AND TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) <= 12 ORDER BY EVENT_TIMESTAMP;"),
    # 7 DERNIERS JOURS
    ("Quelles actions ont ete faites dans les 7 derniers jours ?",
     "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-7) ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Combien d'actions dans les 30 derniers jours ?",
     "SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-30);"),
]
for q, sql in b11_base:
    bloc11.append({"instruction": q, "output": sql})
    for pfx in ["Audit Oracle : ","Question : ","Analyse temporelle : ","Rapport : "]:
        bloc11.append({"instruction": pfx + q, "output": sql})
while len(bloc11) < 600:
    ex = random.choice(bloc11[:120])
    pfx = random.choice(["Hier : ","Ce mois : ","Cette semaine : ","Nuit : ","Temporel : "])
    bloc11.append({"instruction": pfx + ex["instruction"].lower().capitalize(), "output": ex["output"]})
bloc11 = bloc11[:600]

# ══════════════════════════════════════════════════════════════
# BLOC 12 — Questions combinées : qui + quand + quoi — 500 ex
# ══════════════════════════════════════════════════════════════
bloc12 = []
b12_base = [
    ("Qui s'est connecte hier soir dans la base Oracle ?",
     "SELECT DBUSERNAME, EVENT_TIMESTAMP, USERHOST FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' AND TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-1) AND TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= 18 ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a touche la base hier dans la nuit ?",
     "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-1) AND TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= 22 ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a modifie des donnees hier dans la base ?",
     "SELECT DBUSERNAME, ACTION_NAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-1) AND ACTION_NAME IN ('INSERT','UPDATE','DELETE') ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui s'est connecte cette semaine dans la base Oracle ?",
     "SELECT DISTINCT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' AND TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-7) GROUP BY DBUSERNAME ORDER BY NB DESC;"),
    ("Qui a modifie des donnees cette semaine ?",
     "SELECT DBUSERNAME, OBJECT_NAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME IN ('INSERT','UPDATE','DELETE') AND TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-7) GROUP BY DBUSERNAME, OBJECT_NAME ORDER BY NB DESC;"),
    ("Qui a fait le plus de connexions ce mois-ci ?",
     "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' AND TRUNC(EVENT_TIMESTAMP,'MM') = TRUNC(SYSDATE,'MM') GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Est-ce que quelqu'un a fait des choses bizarres la nuit cette semaine ?",
     "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL WHERE TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= 22 AND TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-7) ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Y a-t-il eu des connexions suspectes la nuit cette semaine ?",
     "SELECT DBUSERNAME, EVENT_TIMESTAMP, USERHOST FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' AND TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= 22 AND TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-7) ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Est-ce que quelqu'un a fait des choses bizarres la nuit ce mois-ci ?",
     "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL WHERE TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= 22 AND TRUNC(EVENT_TIMESTAMP,'MM') = TRUNC(SYSDATE,'MM') ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a touche la table EMPLOYEES cette semaine ?",
     "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='EMPLOYEES' AND TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-7) ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a acces a VROMUALD hier ?",
     "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='VROMUALD' AND TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-1) ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a touche PAYROLL ce mois-ci ?",
     "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='PAYROLL' AND TRUNC(EVENT_TIMESTAMP,'MM') = TRUNC(SYSDATE,'MM') ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Y a-t-il eu des suppressions de tables hier ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='DROP TABLE' AND TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-1) ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Y a-t-il eu des creations de comptes cette semaine ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='CREATE USER' AND TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-7) ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a accorde des privileges ce mois-ci ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='GRANT' AND TRUNC(EVENT_TIMESTAMP,'MM') = TRUNC(SYSDATE,'MM') ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Y a-t-il eu des modifications de structure cette semaine ?",
     "SELECT DBUSERNAME, ACTION_NAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME IN ('ALTER TABLE','ALTER USER','DROP TABLE','CREATE TABLE') AND TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-7) ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui a supprime des comptes hier ?",
     "SELECT DBUSERNAME, OBJECT_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='DROP USER' AND TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-1) ORDER BY EVENT_TIMESTAMP DESC;"),
    ("Qui s'est connecte la nuit avant-hier ?",
     "SELECT DBUSERNAME, EVENT_TIMESTAMP, USERHOST FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' AND TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-2) AND TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= 22 ORDER BY EVENT_TIMESTAMP DESC;"),
]
for q, sql in b12_base:
    bloc12.append({"instruction": q, "output": sql})
    for pfx in ["Audit Oracle : ","Securite : ","Analyse : ","Question : ","Combinee : "]:
        bloc12.append({"instruction": pfx + q, "output": sql})
while len(bloc12) < 500:
    ex = random.choice(bloc12[:108])
    pfx = random.choice(["Rapport : ","Combinee : ","Audit : ","Surveillance : "])
    bloc12.append({"instruction": pfx + ex["instruction"].lower().capitalize(), "output": ex["output"]})
bloc12 = bloc12[:500]

# ══════════════════════════════════════════════════════════════
# ASSEMBLAGE FINAL — exactement 7500 exemples
# B1:400 B2:400 B3:400 B4:300 B5:300 B6:400 B7:400
# B8:500 B9:600 B10:400 B11:600 B12:500 = 5700 -> padding +1800
# ══════════════════════════════════════════════════════════════
all_examples = (bloc1 + bloc2 + bloc3 + bloc4 + bloc5 +
                bloc6 + bloc7 + bloc8 + bloc9 + bloc10 +
                bloc11 + bloc12)

random.seed(99)
while len(all_examples) < 7500:
    all_examples.append(random.choice(all_examples))
all_examples = all_examples[:7500]

random.shuffle(all_examples)
df = pd.DataFrame(all_examples)
df.to_csv("oracle_nlp_dataset.csv", index=False)

total = len(all_examples)
print("=" * 65)
print("  DATASET V4 — COLONNES REELLES + DDL/DCL COMPLET")
print("=" * 65)
print(f"  TOTAL    : {total} exemples")
blocs_info = [
    ("B1  - DBA_USERS / utilisateurs existants",              len(bloc1)),
    ("B2  - Plus actif / moins actif (DBUSERNAME)",           len(bloc2)),
    ("B3  - Derniere personne / tracabilite (OBJECT_NAME)",   len(bloc3)),
    ("B4  - Dernieres N actions (EVENT_TIMESTAMP)",           len(bloc4)),
    ("B5  - COUNT DISTINCT + stats",                          len(bloc5)),
    ("B6  - SELECT simples + securite",                       len(bloc6)),
    ("B7  - SQL->FR",                                         len(bloc7)),
    ("B8  - ACTION classiques LOGON/SELECT/INSERT/...",       len(bloc8)),
    ("B9  - ACTION DDL/DCL ALTER/DROP/CREATE/GRANT/... [NOUVEAU]", len(bloc9)),
    ("B10 - OBJ_NAME textuel (OBJECT_NAME reel)",             len(bloc10)),
    ("B11 - Notions de temps (TRUNC(EVENT_TIMESTAMP)) [MIS A JOUR]", len(bloc11)),
    ("B12 - Questions combinees qui+quand+quoi",              len(bloc12)),
    ("PADDING aleatoire",                                     total - 5700),
]
for name, n in blocs_info:
    pct = n / total * 100
    print(f"  {name:<58} {n:>4}  ({pct:>4.1f}%)")
print()
assert total == 7500, f"ERREUR : {total} exemples au lieu de 7500 !"
print("✅ Dataset V4 sauvegarde : oracle_nlp_dataset.csv")
print(f"   {total} exemples — assert OK")


  DATASET V4 — COLONNES REELLES + DDL/DCL COMPLET
  TOTAL    : 7500 exemples
  B1  - DBA_USERS / utilisateurs existants                    400  ( 5.3%)
  B2  - Plus actif / moins actif (DBUSERNAME)                 400  ( 5.3%)
  B3  - Derniere personne / tracabilite (OBJECT_NAME)         400  ( 5.3%)
  B4  - Dernieres N actions (EVENT_TIMESTAMP)                 300  ( 4.0%)
  B5  - COUNT DISTINCT + stats                                300  ( 4.0%)
  B6  - SELECT simples + securite                             400  ( 5.3%)
  B7  - SQL->FR                                               400  ( 5.3%)
  B8  - ACTION classiques LOGON/SELECT/INSERT/...             500  ( 6.7%)
  B9  - ACTION DDL/DCL ALTER/DROP/CREATE/GRANT/... [NOUVEAU]  600  ( 8.0%)
  B10 - OBJ_NAME textuel (OBJECT_NAME reel)                   400  ( 5.3%)
  B11 - Notions de temps (TRUNC(EVENT_TIMESTAMP)) [MIS A JOUR]  600  ( 8.0%)
  B12 - Questions combinees qui+quand+quoi                    500  ( 6.7%)
  PADDING aleatoire  

In [8]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 6 — Fine-tuning LoRA V5                             ║
# ║  FIX LOSS=0 : MAX_LEN=1024 > prompt(665) — labels non nuls  ║
# ║  FIX OOM    : batch=1, grad_accum=8, gradient_checkpointing  ║
# ║  FIX MEM    : gc.collect() + empty_cache() au démarrage      ║
# ╚══════════════════════════════════════════════════════════════╝
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import torch, shutil, os, gc

# ── Libérer la mémoire GPU avant de commencer ──────────────────
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"GPU libre au départ : {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

LORA_DIR = "tinyllama_oracle_lora"

SYSTEM_PROMPT = (
    "Tu es un expert Oracle Database specialise en audit SQL.\n"
    "Table principale : ORACLE_AUDIT_TRAIL\n"
    "Colonnes reelles : ID, AUDIT_TYPE, SESSIONID, OS_USERNAME, USERHOST, TERMINAL, "
    "AUTHENTICATION_TYPE, DBUSERNAME, CLIENT_PROGRAM_NAME, OBJECT_SCHEMA, OBJECT_NAME, "
    "SQL_TEXT, SQL_BINDS, EVENT_TIMESTAMP, ACTION_NAME, INSTANCE\n"
    "Vue systeme : DBA_USERS (colonnes : USERNAME, USER_ID, ACCOUNT_STATUS, LOCK_DATE, "
    "EXPIRY_DATE, DEFAULT_TABLESPACE, TEMPORARY_TABLESPACE, CREATED, PROFILE)\n"
    "Regles importantes :\n"
    "- Pour compter les utilisateurs EXISTANTS dans la base -> utiliser DBA_USERS\n"
    "- Pour les evenements/actions/audit -> utiliser ORACLE_AUDIT_TRAIL\n"
    "- Colonne utilisateur = DBUSERNAME (pas DB_USER)\n"
    "- Colonne timestamp = EVENT_TIMESTAMP (pas TIMESTAMP)\n"
    "- Colonne objet = OBJECT_NAME (pas OBJ_NAME)\n"
    "- Colonne hote = USERHOST (pas OS_HOST)\n"
    "- Colonne identifiant = ID (pas AUDIT_ID)\n"
    "- Pour 'derniere personne' -> retourner DBUSERNAME + ACTION_NAME + EVENT_TIMESTAMP\n"
    "- Pour 'toutes actions' -> ne pas filtrer sur ACTION_NAME sauf si demande\n"
    "- FETCH FIRST N ROWS ONLY pour limiter les resultats Oracle\n"
    "- Filtrage temporel par jour : TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-N)\n"
    "- Filtrage temporel par periode : TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-N)\n"
    "- Filtrage temporel par mois : TRUNC(EVENT_TIMESTAMP,'MM') = TRUNC(SYSDATE,'MM')\n"
    "- Filtrage horaire : TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= N\n"
    "- Actions DDL : ALTER TABLE, ALTER USER, DROP TABLE, DROP USER, CREATE TABLE, CREATE USER\n"
    "- Actions DCL : GRANT, REVOKE\n"
    "- Actions DML : SELECT, INSERT, UPDATE, DELETE\n"
    "- Actions session : LOGON, LOGOFF\n"
    "- Autres actions : EXECUTE, TRUNCATE TABLE, LOCK TABLE, RENAME, INDEX\n"
    "Reponds uniquement en SQL Oracle valide."
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Chargement du modele de base...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
)
# Gradient checkpointing — économise ~30% VRAM
base_model.gradient_checkpointing_enable()
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

# ── r=32, alpha=64 — qualité maximale ─────────────────────────
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj","v_proj","k_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
print("  Parametres entrainables :")
model.print_trainable_parameters()

# ── Dataset ────────────────────────────────────────────────────
raw = load_dataset("csv", data_files="oracle_nlp_dataset.csv")["train"]
raw = raw.select(range(len(raw)))
print(f"  Exemples charges : {len(raw)}")
assert len(raw) == 7500, f"ATTENTION : {len(raw)} exemples charges au lieu de 7500"

# ── Template TinyLlama ─────────────────────────────────────────
INST_TEMPLATE = (
    f"<|system|>{SYSTEM_PROMPT}<|end|>"
    "<|user|>{instr}<|end|>"
    "<|assistant|>"
)

# ── FIX LOSS=0 : MAX_LEN=1024 — le prompt réel fait 665 tokens ─
# Diagnostic confirmé : prompt_len=665 > 384 → tous labels=-100
# 1024 couvre prompt(665) + SQL(~60) avec marge confortable
MAX_LEN = 1024

# Vérification au démarrage
sample = raw[0]
prompt_test = INST_TEMPLATE.format(instr=sample["instruction"])
ids_test = tokenizer(prompt_test)["input_ids"]
print(f"  Vérification : prompt_len={len(ids_test)}, MAX_LEN={MAX_LEN}")
assert len(ids_test) < MAX_LEN, f"ERREUR : prompt ({len(ids_test)}) >= MAX_LEN ({MAX_LEN})"
print(f"  ✅ {MAX_LEN - len(ids_test)} tokens disponibles pour la réponse SQL")

def preprocess(examples):
    full_texts = []
    for instr, out in zip(examples["instruction"], examples["output"]):
        prompt = INST_TEMPLATE.format(instr=instr)
        full_texts.append(prompt + str(out) + "<|end|>")

    tok = tokenizer(
        full_texts,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
        add_special_tokens=True,
    )

    labels = []
    for i, (instr, out) in enumerate(zip(examples["instruction"], examples["output"])):
        prompt = INST_TEMPLATE.format(instr=instr)
        prompt_ids = tokenizer(
            prompt,
            truncation=True,
            max_length=MAX_LEN,
            add_special_tokens=True,
        )["input_ids"]
        prompt_len = len(prompt_ids)

        # Masquer le prompt, apprendre uniquement la réponse SQL
        label = [-100] * prompt_len + list(tok["input_ids"][i][prompt_len:])
        label = label[:MAX_LEN]
        while len(label) < MAX_LEN:
            label.append(-100)
        labels.append(label)

    tok["labels"] = labels
    return tok

tokenized = raw.map(preprocess, batched=True, remove_columns=["instruction","output"])
tokenized.set_format(type="torch", columns=["input_ids","attention_mask","labels"])

# ── TrainingArguments — batch=1 pour tenir dans 15.6 GB ───────
# batch_size=1 × grad_accum=8 = batch effectif 8 (identique à 2×4)
training_args = TrainingArguments(
    output_dir=LORA_DIR,
    per_device_train_batch_size=1,       # 2 → 1 pour MAX_LEN=1024
    gradient_accumulation_steps=8,       # 4 → 8 pour garder batch effectif=8
    num_train_epochs=5,
    learning_rate=2e-4,
    fp16=True,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    logging_steps=50,
    save_strategy="epoch",
    report_to=[],
    push_to_hub=False,
    optim="adamw_torch",
    dataloader_num_workers=1,
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
)

print(f"Entrainement V5 : 5 epoques, LoRA r=32, MAX_LEN={MAX_LEN}, {len(raw)} exemples...")
print(f"GPU utilise avant train : {torch.cuda.memory_allocated()/1e9:.2f} GB")
trainer.train()

model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"✅ Adapter LoRA V5 sauvegarde : {LORA_DIR}/")
shutil.make_archive(LORA_DIR, "zip", LORA_DIR)
print(f"📦 Archive : {LORA_DIR}.zip")

GPU libre au départ : 10.36 GB
Chargement du modele de base...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Parametres entrainables :
trainable params: 25,231,360 || all params: 1,125,279,744 || trainable%: 2.2422
  Exemples charges : 7500
  Vérification : prompt_len=676, MAX_LEN=1024
  ✅ 348 tokens disponibles pour la réponse SQL


Map:   0%|          | 0/7500 [00:00<?, ? examples/s]

Entrainement V5 : 5 epoques, LoRA r=32, MAX_LEN=1024, 7500 exemples...
GPU utilise avant train : 9.44 GB


CheckpointError: torch.utils.checkpoint: A different number of tensors was saved during the original forward and recomputation.
Number of tensors saved during forward: 95
Number of tensors saved during recomputation: 64.

Tip: To see a more detailed error message, either pass `debug=True` to
`torch.utils.checkpoint.checkpoint(...)` or wrap the code block
with `with torch.utils.checkpoint.set_checkpoint_debug_enabled(True):` to
enable checkpoint‑debug mode globally.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 7 — Chargement LoRA V5 + fonctions d'inference      ║
# ║  FIX : generate_sql() et generate_fr() sont SEPAREES         ║
# ║  FIX template : <|system|> identique a l'entrainement V5     ║
# ╚══════════════════════════════════════════════════════════════╝
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch, re

LORA_DIR = "tinyllama_oracle_lora"

SYSTEM_PROMPT = (
    "Tu es un expert Oracle Database specialise en audit SQL.\n"
    "Table principale : ORACLE_AUDIT_TRAIL\n"
    "Colonnes reelles : ID, AUDIT_TYPE, SESSIONID, OS_USERNAME, USERHOST, TERMINAL, "
    "AUTHENTICATION_TYPE, DBUSERNAME, CLIENT_PROGRAM_NAME, OBJECT_SCHEMA, OBJECT_NAME, "
    "SQL_TEXT, SQL_BINDS, EVENT_TIMESTAMP, ACTION_NAME, INSTANCE\n"
    "Vue systeme : DBA_USERS (colonnes : USERNAME, USER_ID, ACCOUNT_STATUS, LOCK_DATE, "
    "EXPIRY_DATE, DEFAULT_TABLESPACE, TEMPORARY_TABLESPACE, CREATED, PROFILE)\n"
    "Regles importantes :\n"
    "- Pour compter les utilisateurs EXISTANTS dans la base -> utiliser DBA_USERS\n"
    "- Pour les evenements/actions/audit -> utiliser ORACLE_AUDIT_TRAIL\n"
    "- Colonne utilisateur = DBUSERNAME\n"
    "- Colonne timestamp = EVENT_TIMESTAMP\n"
    "- Colonne objet = OBJECT_NAME\n"
    "- Colonne hote = USERHOST\n"
    "- Colonne identifiant = ID\n"
    "- Pour 'derniere personne' -> retourner DBUSERNAME + ACTION_NAME + EVENT_TIMESTAMP\n"
    "- Pour 'toutes actions' -> ne pas filtrer sur ACTION_NAME sauf si demande\n"
    "- FETCH FIRST N ROWS ONLY pour limiter les resultats Oracle\n"
    "- Filtrage par jour : TRUNC(EVENT_TIMESTAMP) = TRUNC(SYSDATE-N)\n"
    "- Filtrage par periode : TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-N)\n"
    "- Filtrage par mois : TRUNC(EVENT_TIMESTAMP,'MM') = TRUNC(SYSDATE,'MM')\n"
    "- Filtrage horaire : TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= N\n"
    "- Actions DDL : ALTER TABLE, ALTER USER, DROP TABLE, DROP USER, CREATE TABLE, CREATE USER\n"
    "- Actions DCL : GRANT, REVOKE\n"
    "- Actions DML : SELECT, INSERT, UPDATE, DELETE\n"
    "- Actions session : LOGON, LOGOFF\n"
    "Reponds uniquement en SQL Oracle valide."
)

print(f"Chargement modele de base : {MODEL_DIR}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, dtype=torch.float16, device_map="auto"
)
print(f"Chargement adapter LoRA : {LORA_DIR}")
model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.eval()

# ── Post-processing SQL V2 — 4 regles semantiques ──────────────
def post_process_sql(sql: str, question: str) -> str:
    # Fix Unicode >= <=
    sql = sql.replace("≥", ">=").replace("≤", "<=")
    q = question.lower()
    su = sql.upper()
    # REGLE 1 : utilisateurs existants -> DBA_USERS
    kw_dba = ["existent dans la base","crees dans oracle","comptes oracle",
              "utilisateurs oracle existants","nombre d'utilisateurs",
              "nombre de comptes","comptes dans la base","schemas oracle"]
    if any(k in q for k in kw_dba):
        if "ORACLE_AUDIT_TRAIL" in su and "DBA_USERS" not in su:
            sql = re.sub(r"FROM\s+ORACLE_AUDIT_TRAIL", "FROM DBA_USERS", sql, flags=re.IGNORECASE)
            sql = re.sub(r"WHERE\s+ACTION_NAME\s*=\s*'[^']+'\s*", "", sql, flags=re.IGNORECASE)
            sql = re.sub(r"AND\s+ACTION_NAME\s*=\s*'[^']+'", "", sql, flags=re.IGNORECASE)
            sql = re.sub(r"\bDB_USER\b", "DBUSERNAME", sql, flags=re.IGNORECASE)
    sql = re.sub(r"\bOBJ_NAME\b", "OBJECT_NAME", sql, flags=re.IGNORECASE)
    sql = re.sub(r"\bAUDIT_ID\b", "ID", sql, flags=re.IGNORECASE)
    sql = re.sub(r"\bOS_HOST\b", "USERHOST", sql, flags=re.IGNORECASE)
    # Fix Unicode >= <=
    sql = sql.replace("≥", ">=").replace("≤", "<=")
    # REGLE 2 : "derniere personne" -> inclure DB_USER + ACTION_NAME
    kw_last = ["derniere personne","dernier utilisateur","touche en dernier",
               "qui a modifie","modifie en dernier","dernier acces","qui a fait la derniere"]
    if any(k in q for k in kw_last):
        if "DB_USER" not in su and "USERNAME" not in su:
            sql = re.sub(r"SELECT\s+MAX\(EVENT_TIMESTAMP\)",
                         "SELECT DBUSERNAME, ACTION_NAME, MAX(EVENT_TIMESTAMP) AS DERNIERE_ACTION",
                         sql, flags=re.IGNORECASE)
        if "ACTION_NAME='SELECT'" in su.replace(" ",""):
            sql = re.sub(r"AND\s+ACTION_NAME\s*=\s*'SELECT'", "", sql, flags=re.IGNORECASE)
            sql = re.sub(r"WHERE\s+ACTION_NAME\s*=\s*'SELECT'\s*AND\s*", "WHERE ", sql, flags=re.IGNORECASE)
            sql = re.sub(r"WHERE\s+ACTION_NAME\s*=\s*'SELECT'\s*$", "", sql, flags=re.IGNORECASE)
    # REGLE 3 : "toutes actions" -> retirer filtre ACTION_NAME=SELECT
    kw_all = ["le plus d'actions","le plus d'operations","le plus interagi","toutes actions","toutes les actions"]
    if any(k in q for k in kw_all):
        sql = re.sub(r"WHERE\s+ACTION_NAME\s*=\s*'SELECT'\s*AND\s*", "WHERE ", sql, flags=re.IGNORECASE)
        sql = re.sub(r"AND\s+ACTION_NAME\s*=\s*'SELECT'", "", sql, flags=re.IGNORECASE)
        sql = re.sub(r"WHERE\s+ACTION_NAME\s*=\s*'SELECT'\s*$", "", sql, flags=re.IGNORECASE)
    # REGLE 4 : "modifie des tables" -> filtrer INSERT/UPDATE/DELETE
    kw_modif = ["modifie le plus de tables","tables differentes","tables modifiees",
                "modifications","le plus de tables","tables distinctes modifiees"]
    if any(k in q for k in kw_modif):
        if "INSERT" not in su and "UPDATE" not in su and "DELETE" not in su:
            if "WHERE" in su:
                sql = re.sub(r"WHERE\s+", "WHERE ACTION_NAME IN ('INSERT','UPDATE','DELETE') AND ",
                             sql, count=1, flags=re.IGNORECASE)
            elif "GROUP BY" in su:
                sql = re.sub(r"GROUP BY",
                             "WHERE ACTION_NAME IN ('INSERT','UPDATE','DELETE') GROUP BY",
                             sql, count=1, flags=re.IGNORECASE)
    if ";" in sql:
        sql = sql[:sql.index(";")+1]
    return sql.strip()

def _call_model(prompt: str, max_new_tokens: int = 150, temperature: float = 0.1) -> str:
    """Appel brut au modele, retourne le texte apres <|assistant|>."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=(temperature > 0),
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.15,
        )
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    # Extraire uniquement la réponse après <|assistant|>
    if "<|assistant|>" in decoded:
        decoded = decoded.split("<|assistant|>")[-1]
    if "<|end|>" in decoded:
        decoded = decoded.split("<|end|>")[0]
    return decoded.strip()

def generate_sql(question: str, max_new_tokens: int = 120) -> str:
    """
    FR -> SQL Oracle.
    Utilise le template TinyLlama <|system|> identique a l'entrainement.
    """
    prompt = (
        f"<|system|>{SYSTEM_PROMPT}<|end|>"
        f"<|user|>{question}<|end|>"
        f"<|assistant|>"
    )
    raw = _call_model(prompt, max_new_tokens=max_new_tokens, temperature=0.1)
    return post_process_sql(raw, question)

def generate_fr(sql_query: str, sqlplus_result: str = "", max_new_tokens: int = 180) -> str:
    """
    SQL + resultat -> Francais naturel.
    FIX : utilise le prefixe EXACT du dataset bloc7 pour activer les poids SQL->FR.
    N'applique PAS post_process_sql (ce n'est pas du SQL en sortie).
    """
    if sqlplus_result:
        instruction = (
            f"Traduis ce SQL Oracle en francais : {sql_query}\n"
            f"Resultat Oracle :\n{sqlplus_result}"
        )
    else:
        instruction = f"Traduis ce SQL Oracle en francais : {sql_query}"
    prompt = (
        f"<|system|>{SYSTEM_PROMPT}<|end|>"
        f"<|user|>{instruction}<|end|>"
        f"<|assistant|>"
    )
    return _call_model(prompt, max_new_tokens=max_new_tokens, temperature=0.3)

# Alias retrocompatibilite cellule 9 (generate = generate_sql)
def generate(prompt: str, max_new_tokens: int = 200, temperature: float = 0.1) -> str:
    full_prompt = (
        f"<|system|>{SYSTEM_PROMPT}<|end|>"
        f"<|user|>{prompt}<|end|>"
        f"<|assistant|>"
    )
    raw = _call_model(full_prompt, max_new_tokens=max_new_tokens, temperature=temperature)
    return post_process_sql(raw, prompt)

print("✅ Modele V5 pret pour les tests interactifs.")
print(f"   Device : {next(model.parameters()).device}")
print("   generate_sql()  -> FR vers SQL Oracle (post-processing actif)")
print("   generate_fr()   -> SQL + resultat vers Francais (FIX V3)")
print("   generate()      -> alias retrocompat pour cellule 9")


Chargement modele de base : TinyLlama-1.1B-Chat-v1.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Chargement adapter LoRA : tinyllama_oracle_lora
✅ Modele V2 pret pour les tests interactifs.
   Device : cuda:0
   generate_sql()  -> FR vers SQL Oracle (post-processing actif)
   generate_fr()   -> SQL + resultat vers Francais (FIX V2.1)
   generate()      -> alias retrocompat pour cellule 9


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 8 — Apercu detaille de la table ORACLE_AUDIT_TRAIL  ║
# ╚══════════════════════════════════════════════════════════════╝
import pandas as pd

AUDIT_DF = pd.read_csv("oracle_audit_trail.csv")

print("╔══════════════════════════════════════════════════════════╗")
print("║         ORACLE_AUDIT_TRAIL — REFERENCE COMPLETE          ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Lignes : {len(AUDIT_DF):<5}  |  Colonnes : {len(AUDIT_DF.columns):<5}               ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  COLONNES INTERROGEABLES :                                ║")
cols_info = [
    ("ID",             "INT",      "Cle primaire auto-incrementee"),
    ("EVENT_TIMESTAMP","TIMESTAMP","Horodatage evenement"),
    ("OS_USERNAME",    "VARCHAR2", "Utilisateur Linux/OS"),
    ("DBUSERNAME",     "VARCHAR2", "Utilisateur Oracle"),
    ("USERHOST",       "VARCHAR2", "Serveur source"),
    ("TERMINAL",       "VARCHAR2", "Terminal (pts/0, console...)"),
    ("SESSIONID",      "INT",      "ID session Oracle"),
    ("ACTION_NAME",    "VARCHAR2", "SELECT/LOGON/ALTER TABLE/..."),
    ("OBJECT_SCHEMA",  "VARCHAR2", "Schema Oracle"),
    ("OBJECT_NAME",    "VARCHAR2", "Nom objet/table"),
    ("SQL_TEXT",       "CLOB",     "Requete SQL executee"),
    ("RETURNCODE",     "INT",      "0=OK, >0=erreur ORA-"),
    ("PRIVILEGE_USED", "VARCHAR2", "Privilege Oracle utilise"),
    ("STATEMENT",      "INT",      "Numero de statement"),
    ("COMMENT_TEXT",   "VARCHAR2", "Commentaire d'audit"),
]
for col, typ, desc in cols_info:
    print(f"║    {col:<20} {typ:<6} {desc:<28}║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  VALEURS DISTINCTES PAR COLONNE CLEF :                   ║")
print(f"║    ACTION_NAME  : {sorted(AUDIT_DF.ACTION_NAME.unique())[:5]}...")
print(f"║    DBUSERNAME   : {sorted(AUDIT_DF.DBUSERNAME.unique())[:5]}...")
print(f"║    USERHOST     : {sorted(AUDIT_DF.USERHOST.unique())}")
print(f"║    RETURNCODE   : {sorted(AUDIT_DF.RETURNCODE.unique())}")
print(f"║    OBJECT_NAME (top 5) : {list(AUDIT_DF.OBJECT_NAME.value_counts().head(5).index)}")
print("╠══════════════════════════════════════════════════════════╣")
print("║  EXEMPLES DE QUESTIONS POSSIBLES :                       ║")
questions_exemples = [
    "Quels utilisateurs Oracle ont effectue des SELECT ?",
    "Combien de connexions LOGON ont ete enregistrees ?",
    "Quels evenements ont un code retour d'erreur (!=0) ?",
    "Quels serveurs ont le plus d'activite d'audit ?",
    "Activite nocturne suspecte (apres 22h) ?",
    "Quels utilisateurs ont acces a DBA_USERS ?",
    "Resume global de l'audit Oracle.",
    "Qui a effectue des operations DELETE ou UPDATE ?",
]
for i, q in enumerate(questions_exemples, 1):
    print(f"║    {i}. {q:<50}║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  5 LIGNES D'APERCU :                                     ║")
print("╚══════════════════════════════════════════════════════════╝")
print(AUDIT_DF[["ID","EVENT_TIMESTAMP","DBUSERNAME","ACTION_NAME",
                "OBJECT_NAME","RETURNCODE"]].head(5).to_string(index=False))


╔══════════════════════════════════════════════════════════╗
║         ORACLE_AUDIT_TRAIL — REFERENCE COMPLETE          ║
╠══════════════════════════════════════════════════════════╣
║  Lignes : 500    |  Colonnes : 15                  ║
╠══════════════════════════════════════════════════════════╣
║  COLONNES INTERROGEABLES :                                ║
║    AUDIT_ID             INT    Cle primaire auto-incrementee║
║    TIMESTAMP            CHAR   Format YYYY-MM-DD HH:MM:SS  ║
║    OS_USER              CHAR   Utilisateur Linux/OS        ║
║    DB_USER              CHAR   Utilisateur Oracle          ║
║    OS_HOST              CHAR   Serveur source              ║
║    TERMINAL             CHAR   Terminal (pts/0, console...)║
║    SESSIONID            INT    ID session Oracle           ║
║    ACTION_NAME          CHAR   SELECT/LOGON/UPDATE/...     ║
║    OBJ_SCHEMA           CHAR   Schema Oracle               ║
║    OBJ_NAME             CHAR   Nom table/vue               ║
║    SQL

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 9 — TEST FR -> SQL : scoring + EXECUTION REELLE     ║
# ╚══════════════════════════════════════════════════════════════╝
import re, pandas as pd
from datetime import datetime, timedelta

AUDIT_DF = pd.read_csv("oracle_audit_trail.csv").fillna("")

def simulate_sql_on_audit(sql: str, df: pd.DataFrame) -> str:
    sql_up = sql.upper().strip()
    if sql_up.startswith("--") or "REFUS" in sql_up:
        return "[REFUS DE SECURITE - comportement attendu]"
    if "DBA_USERS" in sql_up and "ORACLE_AUDIT_TRAIL" not in sql_up:
        dba_users_sim = pd.DataFrame({
            "USERNAME": sorted(df["DBUSERNAME"].unique()),
            "ACCOUNT_STATUS": ["OPEN"] * len(df["DBUSERNAME"].unique()),
            "CREATED": ["2024-01-01"] * len(df["DBUSERNAME"].unique()),
        })
        if "COUNT(*)" in sql_up:
            return f"NB_UTILISATEURS / NB_COMPTES : {len(dba_users_sim)} (utilisateurs distincts dans la base)"
        else:
            return dba_users_sim.head(5).to_string(index=False)
    fetch_match = re.search(r"FETCH FIRST (\d+) ROWS ONLY", sql_up)
    n_rows = int(fetch_match.group(1)) if fetch_match else None
    try:
        result_df = df.copy()
        obj_match = re.search(r"OBJECT_NAME\s*=\s*'([^']+)'", sql, re.IGNORECASE)
        if obj_match:
            result_df = result_df[result_df["OBJECT_NAME"] == obj_match.group(1)]
        # FIX : strip("'\"") corrige le SyntaxError original
        action_in_match = re.search(r"ACTION_NAME\s+IN\s*\(([^)]+)\)", sql, re.IGNORECASE)
        if action_in_match:
            actions = [a.strip().strip("'\"") for a in action_in_match.group(1).split(",")]
            result_df = result_df[result_df["ACTION_NAME"].isin(actions)]
        else:
            action_eq_match = re.search(r"ACTION_NAME\s*=\s*'([^']+)'", sql, re.IGNORECASE)
            if action_eq_match:
                result_df = result_df[result_df["ACTION_NAME"] == action_eq_match.group(1)]
        if "RETURNCODE!=0" in sql_up or "RETURNCODE != 0" in sql_up:
            result_df = result_df[result_df["RETURNCODE"] != 0]
        elif re.search(r"RETURNCODE\s*=\s*(\d+)", sql_up):
            rc_val = int(re.search(r"RETURNCODE\s*=\s*(\d+)", sql_up).group(1))
            result_df = result_df[result_df["RETURNCODE"] == rc_val]
        sysdate_match = re.search(r"SYSDATE-(\d+)", sql_up)
        if sysdate_match:
            days_back = int(sysdate_match.group(1))
            cutoff = (datetime.now() - timedelta(days=days_back)).strftime("%Y-%m-%d")
            result_df = result_df[result_df["EVENT_TIMESTAMP"].str[:10] >= cutoff]
        if "TO_NUMBER(SUBSTR(TIMESTAMP,12,2)) >= 22" in sql_up or ">= 22" in sql_up:
            result_df["_HEURE"] = pd.to_numeric(result_df["EVENT_TIMESTAMP"].str[11:13], errors="coerce")
            result_df = result_df[result_df["_HEURE"] >= 22]
        if "COUNT(DISTINCT OBJECT_NAME)" in sql_up or "COUNT(DISTINCT OBJ_NAME)" in sql_up:
            if "GROUP BY DBUSERNAME" in sql_up or "GROUP BY DB_USER" in sql_up:
                gcol = "DBUSERNAME" if "DBUSERNAME" in result_df.columns else "DB_USER"
                r = result_df.groupby(gcol)["OBJECT_NAME"].nunique().reset_index()
                r.columns = [gcol,"NB_TABLES"]
                r = r.sort_values("NB_TABLES", ascending=False)
                if n_rows: r = r.head(n_rows)
                return r.to_string(index=False)
            else:
                return f"NB_TABLES : {result_df['OBJECT_NAME'].nunique() if 'OBJECT_NAME' in result_df.columns else 'N/A'}"
        if "COUNT(DISTINCT DBUSERNAME)" in sql_up or "COUNT(DISTINCT DB_USER)" in sql_up:
            gcol = "DBUSERNAME" if "DBUSERNAME" in result_df.columns else "DB_USER"
            return f"NB_UTILISATEURS : {result_df[gcol].nunique()}"
        if "COUNT(*)" in sql_up and "GROUP BY" not in sql_up:
            if "SUM(CASE" in sql_up:
                total = len(result_df)
                succes = len(result_df[result_df["RETURNCODE"]==0])
                erreurs = total - succes
                return f"TOTAL: {total} | SUCCES: {succes} | ERREURS: {erreurs}"
            return f"COUNT(*) : {len(result_df)}"
        if ("GROUP BY DBUSERNAME" in sql_up or "GROUP BY DB_USER" in sql_up) and "COUNT(*)" in sql_up:
            gcol = "DBUSERNAME" if "DBUSERNAME" in result_df.columns else "DB_USER"
            r = result_df.groupby(gcol).size().reset_index(name="NB")
            r = r.sort_values("NB", ascending=False)
            if n_rows: r = r.head(n_rows)
            return r.to_string(index=False)
        if "GROUP BY ACTION_NAME" in sql_up:
            r = result_df.groupby("ACTION_NAME").size().reset_index(name="NB")
            r = r.sort_values("NB", ascending=False)
            return r.to_string(index=False)
        if "ORDER BY EVENT_TIMESTAMP DESC" in sql_up or "ORDER BY TIMESTAMP DESC" in sql_up:
            tcol = "EVENT_TIMESTAMP" if "EVENT_TIMESTAMP" in result_df.columns else "TIMESTAMP"
            result_df = result_df.sort_values(tcol, ascending=False)
            if n_rows: result_df = result_df.head(n_rows)
            cols = [c for c in ["ID","EVENT_TIMESTAMP","DBUSERNAME","ACTION_NAME","OBJECT_NAME"] if c in result_df.columns]
            return result_df[cols].to_string(index=False)
        if "DISTINCT DBUSERNAME" in sql_up or "DISTINCT DB_USER" in sql_up:
            gcol = "DBUSERNAME" if "DBUSERNAME" in result_df.columns else "DB_USER"
            users = sorted(result_df[gcol].unique())
            return gcol + " : " + ", ".join(users[:10])
        cols = [c for c in ["ID","EVENT_TIMESTAMP","DBUSERNAME","ACTION_NAME","OBJECT_NAME","RETURNCODE"] if c in result_df.columns]
        tcol = "EVENT_TIMESTAMP" if "EVENT_TIMESTAMP" in result_df.columns else "TIMESTAMP"
        result_df = result_df.sort_values(tcol, ascending=False)
        if n_rows: result_df = result_df.head(n_rows)
        return result_df[cols].head(5).to_string(index=False)
    except Exception as e:
        return f"[Simulation: {str(e)[:80]}]"

REFERENCE_TESTS = [
    {"question": "Combien d'utilisateurs existent dans la base Oracle ?",
     "expected_keywords": ["COUNT","DBA_USERS"], "difficulty": "Q1-Existants",
     "cible": "SELECT COUNT(*) AS NB_UTILISATEURS FROM DBA_USERS;"},
    {"question": "Quel utilisateur a effectue le plus d'actions sur la table DBA_USERS ?",
     "expected_keywords": ["DBUSERNAME","COUNT","OBJECT_NAME","DBA_USERS","GROUP BY","ORDER BY"],
     "difficulty": "Q2-Actif",
     "cible": "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='DBA_USERS' GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"},
    {"question": "Quelle est la derniere personne a avoir touche a la table CUSTOMERS ?",
     "expected_keywords": ["DBUSERNAME","CUSTOMERS","EVENT_TIMESTAMP","DESC"],
     "difficulty": "Q3-DernierePersonne",
     "cible": "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='CUSTOMERS' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 1 ROWS ONLY;"},
    {"question": "Donne-moi les 5 dernieres actions enregistrees dans la base.",
     "expected_keywords": ["EVENT_TIMESTAMP","DESC","FETCH FIRST 5","ID","DBUSERNAME"],
     "difficulty": "Q4-Limit",
     "cible": "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 5 ROWS ONLY;"},
    {"question": "Quels sont les 3 utilisateurs qui ont modifie le plus de tables differentes dans les 7 derniers jours ?",
     "expected_keywords": ["COUNT","DISTINCT","INSERT","UPDATE","DELETE","SYSDATE-7","GROUP BY","FETCH FIRST 3"],
     "difficulty": "Q5-CountDistinct",
     "cible": "SELECT DBUSERNAME, COUNT(DISTINCT OBJECT_NAME) AS NB_TABLES FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME IN ('INSERT','UPDATE','DELETE') AND TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-7) GROUP BY DBUSERNAME ORDER BY NB_TABLES DESC FETCH FIRST 3 ROWS ONLY;"},
    {"question": "Quels utilisateurs Oracle ont effectue des SELECT dans l'audit ?",
     "expected_keywords": ["SELECT","DBUSERNAME","ACTION_NAME"], "difficulty": "Facile",
     "cible": "SELECT DISTINCT DBUSERNAME FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' ORDER BY DBUSERNAME;"},
    {"question": "Quels evenements d'audit se sont soldes par une erreur Oracle ?",
     "expected_keywords": ["RETURNCODE","0"], "difficulty": "Moyen",
     "cible": "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, ACTION_NAME, RETURNCODE FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE!=0 ORDER BY EVENT_TIMESTAMP;"},
    {"question": "Y a-t-il de l'activite nocturne suspecte apres 22h dans l'audit ?",
     "expected_keywords": ["22","EVENT_TIMESTAMP","TO_CHAR"], "difficulty": "Difficile",
     "cible": "SELECT ID, EVENT_TIMESTAMP, DBUSERNAME, OBJECT_NAME FROM ORACLE_AUDIT_TRAIL WHERE TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= 22 ORDER BY EVENT_TIMESTAMP;"},
    {"question": "Supprime tous les enregistrements d'audit.",
     "expected_keywords": ["REFUS","DELETE"], "difficulty": "Securite",
     "cible": "-- REQUETE REFUSEE : les operations DELETE ne sont pas autorisees."},
    {"question": "Combien de connexions LOGON ont ete enregistrees dans l'audit ?",
     "expected_keywords": ["COUNT","LOGON","ACTION_NAME"], "difficulty": "Facile",
     "cible": "SELECT COUNT(*) AS NB_CONNEXIONS FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON';"},
]

def score_response(response: str, expected_keywords: list) -> dict:
    resp_upper = response.upper()
    matched = [kw for kw in expected_keywords if kw.upper() in resp_upper]
    score = len(matched) / len(expected_keywords) * 100
    return {"score": round(score,1), "matched": matched,
            "missing": [kw for kw in expected_keywords if kw.upper() not in resp_upper]}

print("="*72)
print("  TESTS FR -> SQL V2  |  Scoring + Execution sur AUDIT_DF pandas")
print("="*72)

results = []
scores_by_difficulty = {}

for i, test in enumerate(REFERENCE_TESTS, 1):
    q = test["question"]
    sql_genere = generate_sql(q)
    sc = score_response(sql_genere, test["expected_keywords"])
    sc["question"] = q
    sc["response"] = sql_genere
    sc["cible"] = test["cible"]
    sc["difficulty"] = test["difficulty"]
    results.append(sc)
    scores_by_difficulty[test["difficulty"]] = sc["score"]
    exec_result = simulate_sql_on_audit(sql_genere, AUDIT_DF)
    status = "OK" if sc["score"] >= 70 else ("~" if sc["score"] >= 40 else "X")
    print(f"\n--- Test {i:02d} [{test['difficulty']}]  Score SQL: {sc['score']:.0f}%  {status}")
    print(f"  Q         : {q}")
    print(f"  SQL genere: {sql_genere[:120]}{'...' if len(sql_genere)>120 else ''}")
    print(f"  SQL attendu: {test['cible'][:100]}{'...' if len(test['cible'])>100 else ''}")
    print(f"  RESULTAT  : {str(exec_result)[:200]}")
    if sc["missing"]:
        print(f"  Manquants : {', '.join(sc['missing'])}")

avg = sum(r["score"] for r in results) / len(results)
print(f"\n{'='*72}")
print(f"  SCORE MOYEN FR->SQL : {avg:.1f}%")
print(f"\n  SCORES PAR QUESTION DU RAPPORT (objectif >= 90%) :")
for diff in ["Q1-Existants","Q2-Actif","Q3-DernierePersonne","Q4-Limit","Q5-CountDistinct"]:
    s = scores_by_difficulty.get(diff, 0)
    bar = "X" * int(s//10)
    ok = " <-- OK" if s >= 80 else (" <-- proche" if s >= 60 else " <-- a ameliorer")
    print(f"    {diff:<30} {s:>5.1f}%  {bar}{ok}")
print(f"{'='*72}")

print("\n" + "="*72)
print("  ZONE DE TEST LIBRE")
print("="*72)
ma_question = "Qui a effectue la derniere action sur la table ACCOUNTS ?"
sql_libre = generate_sql(ma_question)
exec_libre = simulate_sql_on_audit(sql_libre, AUDIT_DF)
print(f"  Question  : {ma_question}")
print(f"  SQL genere: {sql_libre}")
print(f"  Resultat  : {str(exec_libre)[:300]}")


  TESTS FR -> SQL V2  |  Scoring + Execution sur AUDIT_DF pandas

--- Test 01 [Q1-Existants]  Score SQL: 100%  OK
  Q         : Combien d'utilisateurs existent dans la base Oracle ?
  SQL genere: SELECT COUNT(*) AS NB_UTILISATEURS FROM DBA_USERS;
  SQL attendu: SELECT COUNT(*) AS NB_UTILISATEURS FROM DBA_USERS;
  RESULTAT  : NB_UTILISATEURS / NB_COMPTES : 10 (utilisateurs distincts dans la base)

--- Test 02 [Q2-Actif]  Score SQL: 100%  OK
  Q         : Quel utilisateur a effectue le plus d'actions sur la table DBA_USERS ?
  SQL genere: SELECT DB_USER, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE OBJ_NAME='DBA_USERS' GROUP BY DB_USER ORDER BY NB DESC FETC...
  SQL attendu: SELECT DB_USER, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE OBJ_NAME='DBA_USERS' GROUP BY DB_USER O...
  RESULTAT  : DB_USER  NB
DBA_OPS   3

--- Test 03 [Q3-DernierePersonne]  Score SQL: 100%  OK
  Q         : Quelle est la derniere personne a avoir touche a la table CUSTOMERS ?
  SQL genere: SELECT DB_USER, ACTION

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 10 — TEST SQL->FR (CORRIGE V3)                    ║
# ║  FIX : utilise generate_fr() avec prefixe matching dataset   ║
# ║  FIX : le prompt inclut SQL + resultat Oracle en une fois     ║
# ╚══════════════════════════════════════════════════════════════╝
import pandas as pd

AUDIT_DF = pd.read_csv("oracle_audit_trail.csv").fillna("")

SQL_TO_FR_TESTS = [
    {"label": "Connexions echouees ORA-01017",
     "sql": "SELECT DISTINCT DBUSERNAME, USERHOST FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE=1017;",
     "sqlplus_result": "DBUSERNAME  USERHOST\n---------  ---------------\nSCOTT      app-server-01\nHR         srv-oracle-02\nAPP_USER   monitor-01",
     "expected_keywords": ["SCOTT","HR","erreur","mot de passe","connexion"],
     "difficulty": "Moyen"},
    {"label": "Utilisateur le plus actif",
     "sql": "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DBUSERNAME ORDER BY NB DESC FETCH FIRST 5 ROWS ONLY;",
     "sqlplus_result": "DBUSERNAME  NB\n---------  --\nSYS        87\nSCOTT      45\nHR         23\nAPP_USER   12\nDBA_OPS     8",
     "expected_keywords": ["SYS","87","actif","utilisateur","actions"],
     "difficulty": "Facile"},
    {"label": "Activite nocturne suspecte (apres 22h)",
     "sql": "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE TO_NUMBER(TO_CHAR(EVENT_TIMESTAMP,'HH24')) >= 22;",
     "sqlplus_result": "DB_USER  ACTION_NAME  TIMESTAMP\nSCOTT    SELECT       2026-02-10 23:14:22\nSYS      LOGON        2026-02-12 22:05:11\nHR       SELECT       2026-02-14 22:47:03",
     "expected_keywords": ["22h","nocturne","SCOTT","SYS","suspect"],
     "difficulty": "Difficile"},
    {"label": "Resume global audit",
     "sql": "SELECT COUNT(*) TOTAL, SUM(CASE WHEN RETURNCODE=0 THEN 1 ELSE 0 END) SUCCES, SUM(CASE WHEN RETURNCODE!=0 THEN 1 ELSE 0 END) ERREURS FROM ORACLE_AUDIT_TRAIL;",
     "sqlplus_result": "TOTAL  SUCCES  ERREURS\n-----  ------  -------\n500    432     68",
     "expected_keywords": ["500","432","68","erreur","succes"],
     "difficulty": "Moyen"},
    {"label": "Objets sensibles consultes",
     "sql": "SELECT DISTINCT DBUSERNAME FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME IN ('DBA_USERS','V$SESSION');",
     "sqlplus_result": "DBUSERNAME\n-------\nSYS\nDBA_OPS\nSCOTT",
     "expected_keywords": ["SYS","DBA_OPS","DBA_USERS","sensible","systeme"],
     "difficulty": "Difficile"},
    {"label": "Top 3 utilisateurs tables modifiees 7 jours",
     "sql": "SELECT DBUSERNAME, COUNT(DISTINCT OBJECT_NAME) AS NB_TABLES FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME IN ('INSERT','UPDATE','DELETE') AND TRUNC(EVENT_TIMESTAMP) >= TRUNC(SYSDATE-7) GROUP BY DBUSERNAME ORDER BY NB_TABLES DESC FETCH FIRST 3 ROWS ONLY;",
     "sqlplus_result": "DBUSERNAME  NB_TABLES\n---------  ---------\nSCOTT              4\nHR                 2\nSYS                1",
     "expected_keywords": ["SCOTT","4","tables","modifie","7 jours"],
     "difficulty": "Difficile"},
    {"label": "Derniere personne sur CUSTOMERS",
     "sql": "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJECT_NAME='CUSTOMERS' ORDER BY EVENT_TIMESTAMP DESC FETCH FIRST 1 ROWS ONLY;",
     "sqlplus_result": "DB_USER     ACTION_NAME  TIMESTAMP\n----------  -----------  -------------------\nREPORT_USR  SELECT       2026-02-24 14:32:11",
     "expected_keywords": ["REPORT_USR","CUSTOMERS","SELECT","2026"],
     "difficulty": "Moyen"},
    {"label": "Distribution des actions Oracle",
     "sql": "SELECT ACTION_NAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY ACTION_NAME ORDER BY NB DESC;",
     "sqlplus_result": "ACTION_NAME     NB\n-----------     --\nSELECT         325\nLOGON           57\nLOGOFF          55\nUPDATE          18\nDELETE          12\nINSERT          12",
     "expected_keywords": ["SELECT","325","LOGON","UPDATE","actions"],
     "difficulty": "Facile"},
]

def score_sqlfr(response: str, expected_keywords: list) -> dict:
    resp_lower = response.lower()
    matched = [kw for kw in expected_keywords if kw.lower() in resp_lower]
    score = len(matched) / len(expected_keywords) * 100
    return {"score": round(score,1), "matched": matched,
            "missing": [kw for kw in expected_keywords if kw.lower() not in resp_lower]}

print("="*72)
print("  TEST : Resultat Oracle brut -> Francais naturel (V3 CORRIGE)")
print("  generate_fr() utilise le prefixe exact du dataset bloc7")
print("="*72)

results_sqlfr = []

for i, test in enumerate(SQL_TO_FR_TESTS, 1):
    # FIX : appel a generate_fr() avec sql + resultat comme appris en bloc7
    reponse = generate_fr(test["sql"], test["sqlplus_result"], max_new_tokens=180)
    sc = score_sqlfr(reponse, test["expected_keywords"])
    results_sqlfr.append({"label": test["label"], "score": sc["score"], "difficulty": test["difficulty"]})
    status = "OK" if sc["score"] >= 60 else ("~" if sc["score"] >= 30 else "X")
    print(f"\n--- Test {i:02d} [{test['difficulty']}]  {status}  Score: {sc['score']:.0f}%  | {test['label']}")
    print(f"  SQL      : {test['sql'][:85]}...")
    print(f"  Resultat :\n{test['sqlplus_result']}")
    print(f"  Traduction : {reponse[:250]}{'...' if len(reponse)>250 else ''}")
    if sc["missing"]:
        print(f"  Manquants  : {', '.join(sc['missing'])}")

avg_sqlfr = sum(r["score"] for r in results_sqlfr) / len(results_sqlfr)
print(f"\n{'='*72}")
print(f"  BILAN SQL->FR (traduction resultats Oracle)")
print("="*72)
for r in results_sqlfr:
    bar = "X" * int(r["score"]//10)
    ok = " OK" if r["score"]>=60 else ("~ proche" if r["score"]>=30 else "")
    print(f"  {r['label']:<45} {r['score']:>5.1f}%  {bar}{ok}")
print(f"  {'─'*55}")
print(f"  Score global SQL->FR : {avg_sqlfr:.1f}%")
print("="*72)

# Zone libre
print("\n" + "="*72)
print("  ZONE LIBRE — Modifiez et re-executez")
print("="*72)
ma_requete_sql = "SELECT ACTION_NAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY ACTION_NAME ORDER BY NB DESC;"
mon_resultat_sqlplus = """ACTION_NAME     NB
-----------     --
SELECT          325
LOGON            57
LOGOFF           55
UPDATE           18
DELETE           12
INSERT           12"""
print(f"  Requete   : {ma_requete_sql}")
print(f"  Resultat Oracle :\n{mon_resultat_sqlplus}")
reponse_libre = generate_fr(ma_requete_sql, mon_resultat_sqlplus, max_new_tokens=200)
print(f"\n  Traduction en francais :\n  {reponse_libre}")


  TEST : Resultat Oracle brut -> Francais naturel (V2.1 CORRIGE)
  generate_fr() utilise le prefixe exact du dataset bloc7

--- Test 01 [Moyen]  X  Score: 0%  | Connexions echouees ORA-01017
  SQL      : SELECT DISTINCT DB_USER, OS_HOST FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE=1017;...
  Resultat :
DB_USER    OS_HOST
---------  ---------------
SCOTT      app-server-01
HR         srv-oracle-02
APP_USER   monitor-01
  Traduction : 
  Manquants  : SCOTT, HR, erreur, mot de passe, connexion

--- Test 02 [Facile]  ~  Score: 40%  | Utilisateur le plus actif
  SQL      : SELECT DB_USER, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DB_USER ORDER BY NB D...
  Resultat :
DB_USER    NB
---------  --
SYS        87
SCOTT      45
HR         23
APP_USER   12
DBA_OPS     8
  Traduction : Les 10 utilisateurs Oracle ayant effectue le plus d'actions sur la table ORACLE_AUDIT_TRAIL sont identifies. Toutes les actions ont ete comptees, pas uniquement les SELECT.
  Manquants  : SYS, 87, actif

--- Test 0

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 10b — Phi-3 mini Q4 local : SQL→FR sans fine-tuning ║
# ║  Télécharge le modèle + zippe pour export local              ║
# ╚══════════════════════════════════════════════════════════════╝
from huggingface_hub import hf_hub_download
import subprocess, os, shutil

# ── 1. Téléchargement Phi-3 mini Q4_K_M (llama.cpp GGUF) ──────
PHI3_DIR  = "phi3-mini-gguf"
PHI3_FILE = "Phi-3-mini-4k-instruct-q4.gguf"
PHI3_PATH = os.path.join(PHI3_DIR, PHI3_FILE)

os.makedirs(PHI3_DIR, exist_ok=True)

if not os.path.exists(PHI3_PATH):
    print("📥 Téléchargement Phi-3 mini Q4_K_M (~2.3 GB)...")
    hf_hub_download(
        repo_id="microsoft/Phi-3-mini-4k-instruct-gguf",
        filename=PHI3_FILE,
        local_dir=PHI3_DIR,
        local_dir_use_symlinks=False,
    )
    print("✅ Phi-3 mini téléchargé.")
else:
    print(f"✅ Phi-3 mini déjà présent : {PHI3_PATH}")

# ── 2. Zipper pour export local ────────────────────────────────
PHI3_ZIP = PHI3_DIR + ".zip"
if not os.path.exists(PHI3_ZIP):
    print(f"📦 Création archive {PHI3_ZIP}...")
    shutil.make_archive(PHI3_DIR, "zip", PHI3_DIR)
    print(f"✅ Archive prête : {PHI3_ZIP}")
else:
    print(f"✅ Archive déjà présente : {PHI3_ZIP}")

# ── 3. Installer llama-cpp-python (inference locale CPU/GPU) ───
print("📦 Installation llama-cpp-python...")
subprocess.run([
    "pip", "install", "-q", "--no-cache-dir",
    "llama-cpp-python==0.2.90",
    "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121"
], check=True)
print("✅ llama-cpp-python installé.")

# ── 4. Chargement du modèle ────────────────────────────────────
from llama_cpp import Llama

print("🔄 Chargement Phi-3 mini en mémoire...")
phi3 = Llama(
    model_path=PHI3_PATH,
    n_ctx=2048,
    n_gpu_layers=35,   # tout sur GPU si dispo, sinon mettre 0 pour CPU
    verbose=False,
)
print("✅ Phi-3 mini prêt.")

# ── 5. Prompt système ──────────────────────────────────────────
ORACLE_ERRORS = {
    "ORA-01017": "mot de passe incorrect ou compte inexistant",
    "ORA-00942": "table ou vue inexistante",
    "ORA-01031": "privileges insuffisants",
    "ORA-00955": "nom d'objet deja utilise",
    "ORA-00001": "violation de contrainte d'unicite",
    "ORA-00904": "nom de colonne invalide",
    "ORA-01000": "nombre maximum de curseurs ouverts depasse",
}

PHI3_SYSTEM = (
    "Tu es un assistant specialise en audit Oracle Database.\n"
    "Tu recois le resultat brut d'une requete SQL executee sur la table ORACLE_AUDIT_TRAIL.\n"
    "Tu traduis ce resultat en francais clair et comprehensible pour un responsable non-technicien.\n"
    "Codes erreur Oracle :\n"
    + "\n".join(f"- {k} : {v}" for k, v in ORACLE_ERRORS.items()) +
    "\nRegles strictes :\n"
    "- Reponds en 2 a 4 phrases maximum\n"
    "- Mentionne les valeurs importantes du resultat (noms, nombres, dates)\n"
    "- Si le resultat est vide, dis-le clairement\n"
    "- Si tu vois un code ORA-, explique ce que cela signifie en francais\n"
    "- Ne montre jamais le SQL brut dans ta reponse\n"
    "- Reponds uniquement en francais\n"
)

def traduire_avec_phi3(sql: str, resultat_brut: str, max_tokens: int = 200) -> str:
    user_msg = (
        f"Requete SQL executee :\n{sql}\n\n"
        f"Resultat Oracle brut :\n{resultat_brut}\n\n"
        f"Traduis ce resultat en francais clair pour un responsable."
    )
    prompt = (
        f"<|system|>\n{PHI3_SYSTEM}<|end|>\n"
        f"<|user|>\n{user_msg}<|end|>\n"
        f"<|assistant|>\n"
    )
    response = phi3(
        prompt,
        max_tokens=max_tokens,
        temperature=0.2,
        repeat_penalty=1.1,
        stop=["<|end|>", "<|user|>"],
    )
    return response["choices"][0]["text"].strip()

# ── 6. Tests SQL→FR avec Phi-3 ─────────────────────────────────
print("\n" + "="*72)
print("  TESTS SQL->FR avec Phi-3 mini Q4 (local, sans fine-tuning)")
print("="*72)

SQL_TO_FR_TESTS_PHI3 = [
    {
        "label": "Utilisateur le plus actif",
        "sql": "SELECT DBUSERNAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DB_USER ORDER BY NB DESC FETCH FIRST 5 ROWS ONLY;",
        "resultat": "DBUSERNAME  NB\n---------  --\nSYS        87\nSCOTT      45\nHR         23\nAPP_USER   12\nDBA_OPS     8",
        "expected_keywords": ["SYS","87","actif","utilisateur","actions"],
    },
    {
        "label": "Connexions echouees ORA-01017",
        "sql": "SELECT DISTINCT DBUSERNAME, USERHOST FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE=1017;",
        "resultat": "DBUSERNAME  USERHOST\n---------  ---------------\nSCOTT      app-server-01\nHR         srv-oracle-02",
        "expected_keywords": ["SCOTT","HR","erreur","mot de passe","connexion"],
    },
    {
        "label": "Resume global audit",
        "sql": "SELECT COUNT(*) TOTAL, SUM(CASE WHEN RETURNCODE=0 THEN 1 ELSE 0 END) SUCCES, SUM(CASE WHEN RETURNCODE!=0 THEN 1 ELSE 0 END) ERREURS FROM ORACLE_AUDIT_TRAIL;",
        "resultat": "TOTAL  SUCCES  ERREURS\n-----  ------  -------\n500    432     68",
        "expected_keywords": ["500","432","68","erreur","succes"],
    },
    {
        "label": "Activite nocturne suspecte",
        "sql": "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE TO_NUMBER(SUBSTR(TIMESTAMP,12,2)) >= 22;",
        "resultat": "DB_USER  ACTION_NAME  TIMESTAMP\nSCOTT    SELECT       2026-02-10 23:14:22\nSYS      LOGON        2026-02-12 22:05:11",
        "expected_keywords": ["SCOTT","SYS","nuit","suspect","22"],
    },
    {
        "label": "Derniere personne sur CUSTOMERS",
        "sql": "SELECT DBUSERNAME, ACTION_NAME, EVENT_TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE OBJ_NAME='CUSTOMERS' ORDER BY TIMESTAMP DESC FETCH FIRST 1 ROWS ONLY;",
        "resultat": "DB_USER     ACTION_NAME  TIMESTAMP\nREPORT_USR  SELECT       2026-02-24 14:32:11",
        "expected_keywords": ["REPORT_USR","CUSTOMERS","SELECT","2026"],
    },
    {
        "label": "Distribution des actions",
        "sql": "SELECT ACTION_NAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY ACTION_NAME ORDER BY NB DESC;",
        "resultat": "ACTION_NAME  NB\nSELECT       325\nLOGON         57\nLOGOFF        55\nUPDATE        18\nDELETE        12\nINSERT        12",
        "expected_keywords": ["SELECT","325","LOGON","actions"],
    },
    {
        "label": "Resultat vide",
        "sql": "SELECT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='DROP' AND RETURNCODE=0;",
        "resultat": "Aucune ligne retournee.",
        "expected_keywords": ["aucun","vide","aucune","resultat","trouve"],
    },
    {
        "label": "Erreur ORA dans le resultat",
        "sql": "SELECT * FROM DBA_USERS;",
        "resultat": "ORA-00942: table or view does not exist",
        "expected_keywords": ["erreur","inexistant","ORA","privileges","table"],
    },
]

def score_phi3(response: str, keywords: list) -> dict:
    r = response.lower()
    matched = [k for k in keywords if k.lower() in r]
    return {
        "score": round(len(matched)/len(keywords)*100, 1),
        "matched": matched,
        "missing": [k for k in keywords if k.lower() not in r],
    }

results_phi3 = []
for i, test in enumerate(SQL_TO_FR_TESTS_PHI3, 1):
    reponse = traduire_avec_phi3(test["sql"], test["resultat"])
    sc = score_phi3(reponse, test["expected_keywords"])
    results_phi3.append({"label": test["label"], "score": sc["score"]})
    status = "✅" if sc["score"] >= 60 else ("~" if sc["score"] >= 30 else "❌")
    print(f"\n--- Test {i:02d} {status}  Score: {sc['score']:.0f}%  | {test['label']}")
    print(f"  Résultat brut : {test['resultat'][:80]}...")
    print(f"  Traduction    : {reponse[:250]}")
    if sc["missing"]:
        print(f"  Manquants     : {', '.join(sc['missing'])}")

avg_phi3 = sum(r["score"] for r in results_phi3) / len(results_phi3)
print(f"\n{'='*72}")
print(f"  BILAN SQL->FR Phi-3 mini Q4")
print("="*72)
for r in results_phi3:
    bar = "█" * int(r["score"]//10)
    ok = " ✅" if r["score"] >= 60 else ""
    print(f"  {r['label']:<45} {r['score']:>5.1f}%  {bar}{ok}")
print(f"  {'─'*55}")
print(f"  Score global SQL->FR Phi-3 : {avg_phi3:.1f}%")
print("="*72)

# Mise a jour de results_sqlfr pour le bilan cellule 11
results_sqlfr = [{"label": r["label"], "score": r["score"], "difficulty": "Phi3"} for r in results_phi3]
print("\n✅ results_sqlfr mis a jour — cellule 11 utilisera les scores Phi-3")

📥 Téléchargement Phi-3 mini Q4_K_M (~2.3 GB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Phi-3-mini-4k-instruct-q4.gguf:   0%|          | 0.00/2.39G [00:00<?, ?B/s]

✅ Phi-3 mini téléchargé.
📦 Création archive phi3-mini-gguf.zip...
✅ Archive prête : phi3-mini-gguf.zip
📦 Installation llama-cpp-python...
✅ llama-cpp-python installé.
🔄 Chargement Phi-3 mini en mémoire...
✅ Phi-3 mini prêt.

  TESTS SQL->FR avec Phi-3 mini Q4 (local, sans fine-tuning)

--- Test 01 ✅  Score: 60%  | Utilisateur le plus actif
  Résultat brut : DB_USER    NB
---------  --
SYS        87
SCOTT      45
HR         23
APP_USER  ...
  Traduction    : Le résultat de votre requête montre les utilisateurs ayant le plus d'activités dans l'historique des audits Oracle, classés par ordre décroissant. Voici les informations clés :

- L'utilisateur système (SYS) a effectué 87 activités.
- L'utilisateur S
  Manquants     : actif, actions

--- Test 02 ✅  Score: 60%  | Connexions echouees ORA-01017
  Résultat brut : DB_USER    OS_HOST
---------  ---------------
SCOTT      app-server-01
HR       ...
  Traduction    : Le résultat de votre requête montre que deux utilisateurs ont eu des erreu

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 11 — BILAN GLOBAL DE PERFORMANCE DU MODELE          ║
# ╚══════════════════════════════════════════════════════════════╝
import pandas as pd
from datetime import datetime

try:
    scores_fr_sql = {r["difficulty"]: r["score"] for r in results}
    avg_fr_sql    = sum(r["score"] for r in results) / len(results)
    avg_sql_fr    = sum(r["score"] for r in results_sqlfr) / len(results_sqlfr)
    score_global  = (avg_fr_sql + avg_sql_fr) / 2
except:
    avg_fr_sql = avg_sql_fr = score_global = 0.0
    print("⚠️  Executez d'abord les cellules 9 et 10.")

print("╔══════════════════════════════════════════════════════════╗")
print("║          RAPPORT DE PERFORMANCE — MODELE NLP->SQL        ║")
print(f"║          Genere le : {datetime.now().strftime('%Y-%m-%d %H:%M')}                    ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  CONFIGURATION                                            ║")
print(f"║    Modele base     : TinyLlama-1.1B-Chat-v1.0            ║")
print(f"║    Methode         : LoRA Fine-Tuning V5 (r=32, max_len=384)║")
print(f"║    Dataset         : 7500 ex. (3800 FR->SQL + 700 SQL->FR)║")
print(f"║    Epochs          : 5  |  LoRA r=32, alpha=64, max_len=384║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  RESULTATS                                                ║")
print(f"║    FR -> SQL Oracle : {avg_fr_sql:>5.1f}%                             ║")
print(f"║    SQL/Resultat->FR : {avg_sql_fr:>5.1f}%                             ║")
print(f"║    Score global    : {score_global:>5.1f}%                             ║")

if score_global >= 80:
    verdict = "🏆 EXCELLENT — Pret pour production"
    details = "Le modele genere du SQL Oracle precis et des reformulations naturelles."
elif score_global >= 65:
    verdict = "✅ BON — Utilisable avec supervision"
    details = "Performances satisfaisantes. Quelques cas limites a ameliorer."
elif score_global >= 45:
    verdict = "⚠️  MOYEN — Dataset plus large recommande"
    details = "Le modele comprend les requetes simples mais echoue sur les cas complexes."
else:
    verdict = "❌ INSUFFISANT — Entrainement supplementaire necessaire"
    details = "Augmenter le dataset et les epochs d'entrainement."

print(f"║  VERDICT   : {verdict:<44}║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  SCORES PAR QUESTION DU RAPPORT :                        ║")
for diff in ["Q1-Existants","Q2-Actif","Q3-DernierePersonne","Q4-Limit","Q5-CountDistinct"]:
    s = scores_fr_sql.get(diff, 0) if 'scores_fr_sql' in dir() else 0
    ok = "✅" if s >= 80 else ("~" if s >= 60 else "❌")
    print(f"║    {ok} {diff:<30} {s:>5.1f}%              ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  PROCHAINES ETAPES VERS PRODUCTION                       ║")
steps = [
    "1. Quantization 4-bit (bitsandbytes) pour CPU",
    "2. Export GGUF (llama.cpp) pour deploiement leger",
    "3. API FastAPI + validation SQL (SELECT uniquement)",
    "4. Interface Streamlit QueryFlow (app_oracle_nlp.py)",
    "5. Tests sur vraie base Oracle (cx_Oracle / SQLAlchemy)",
    "6. Monitoring et logs d'audit des requetes generees",
]
for step in steps:
    print(f"║    {step:<54}║")
print("╚══════════════════════════════════════════════════════════╝")

# Export CSV
report_data = []
try:
    for r in results:
        report_data.append({"sens":"FR->SQL","test":r["question"][:60],
                             "score":r["score"],"difficulte":r["difficulty"]})
    for r in results_sqlfr:
        report_data.append({"sens":"SQL->FR","test":r["label"],
                             "score":r["score"],"difficulte":r["difficulty"]})
    pd.DataFrame(report_data).to_csv("rapport_performance_modele.csv", index=False)
    print("\n📄 Rapport exporte : rapport_performance_modele.csv")
except:
    pass


╔══════════════════════════════════════════════════════════╗
║          RAPPORT DE PERFORMANCE — MODELE NLP->SQL        ║
║          Genere le : 2026-03-04 14:31                    ║
╠══════════════════════════════════════════════════════════╣
║  CONFIGURATION                                            ║
║    Modele base     : TinyLlama-1.1B-Chat-v1.0            ║
║    Methode         : LoRA Fine-Tuning V2.1 (CORRIGE)     ║
║    Dataset         : 4500 ex. (3800 FR->SQL + 700 SQL->FR)║
║    Epochs          : 5  |  LoRA r=32, alpha=64           ║
╠══════════════════════════════════════════════════════════╣
║  RESULTATS                                                ║
║    FR -> SQL Oracle : 100.0%                             ║
║    SQL/Resultat->FR :  64.4%                             ║
║    Score global    :  82.2%                             ║
║  VERDICT   : 🏆 EXCELLENT — Pret pour production          ║
╠══════════════════════════════════════════════════════════╣
║  SCORES PAR QUESTION

## Deploiement Local & Production

### Fichiers a recuperer depuis Colab

| Fichier | Description |
|---|---|
| `TinyLlama-1.1B-Chat-v1.0.zip` | Modele de base |
| `tinyllama_oracle_lora.zip` | Adapter LoRA V5 fine-tune (r=32, max_len=384) |
| `oracle_audit_trail.csv` | Table d'audit Oracle simulee (500 lignes) |
| `oracle_nlp_dataset.csv` | Dataset V4 (7500 exemples corriges) |
| `rapport_performance_modele.csv` | Resultats de performance |
| `app_oracle_nlp.py` | Interface Streamlit QueryFlow |

### Commandes pip (environnement local)

```bash
python -m venv venv_oracle
source venv_oracle/bin/activate  # Linux/Mac

pip install transformers>=4.40.0 peft>=0.10.0 accelerate>=0.28.0
pip install torch>=2.2.0 pandas>=2.0.0 streamlit pandasql
```

### Lancer l'interface QueryFlow

```bash
streamlit run app_oracle_nlp.py
```

### Architecture de production

```
Utilisateur (question FR)
       ↓
Interface QueryFlow (Streamlit)
       ↓
generate_sql()  →  TinyLlama + LoRA V4
       ↓
post_process_sql() — 4 regles semantiques
       ↓
Oracle DB (cx_Oracle, compte lecture seule)
       ↓
generate_fr()  →  SQL + resultat -> Francais
       ↓
Reponse claire pour le responsable
```

### Corrections V3 appliquees

| Cellule | Probleme V2.0 | Fix V3 |
|---|---|---|
| 5 | Blocs 4 et 7 tronques (total = 4000) | Quotas explicites -> total = 7500 |
| 6 | `load_dataset` charge 4000 ex. | `raw.select(range(len(raw)))` force 7500 |
| 7 | `generate()` appelait `post_process_sql` sur FR | Deux fonctions : `generate_sql()` / `generate_fr()` |
| 10 | Prompt sans prefixe matching | Prefixe `"Traduis ce SQL Oracle en francais : "` exact |
| 9 | `strip("'\"")` SyntaxError | Corrige |
